# Reproducing Yu et al. (2016) from dandiset 000005

Paper: Yu J, Gutnisky DA, Hires SA, Svoboda K (2016). Layer 4 fast-spiking interneurons filter
thalamocortical signals during active somatosensation. *Nature Neuroscience* 19:1647-1657,
[doi:10.1038/nn.4412](https://doi.org/10.1038/nn.4412).

Dandiset: [DANDI:000005](https://dandiarchive.org/dandiset/000005/0.220126.1853), version
0.220126.1853. The paper cites no dandiset version (it predates the archive). 0.220126.1853 is the
only published version, so it is pinned; the draft (last modified 2026-02-12) lists the same 148
asset paths and sizes (printed below). The version is the same as that of the earlier attempt,
`000005-yu-2016-01`.

Claims: `claims/000005/10.1038-nn.4412.yaml`, written by a reader from the version of record with its
Online Methods and the supplement (Supplementary Figures 1 to 14, Supplementary Tables 1 and 2), and
checked by a second reader on 2026-10-20. It replaces the claims file of the earlier attempt, which
was written from the abstract. The ids of eight earlier claims are kept; their criteria are new.
The paper's data statement points to CRCNS ssc-7; no analysis code was found.

This is a repeat attempt. The earlier attempt's findings are checked again in the data check.

Outside input: Supplementary Table 1 of the paper (the mouse line of each mouse), read at run time
from the supplement PDF on the publisher's site. It is used to name the mouse lines that the files
leave blank and to compare the cells per line with Supplementary Table 2.

| Claim | Tested |
|---|---|
| `behavior-touches` | yes |
| `vpm-touch` | yes |
| `vpm-whisking` | yes |
| `vpm-whisking-phase` | yes |
| `exc-vm-touch-whisking` | yes |
| `exc-threshold-touch-vs-whisking` | yes |
| `exc-effective-reversal` | yes |
| `exc-inhibition-timing` | blocked: data missing |
| `exc-inhibition-whisking` | blocked: data missing |
| `fs-touch-response` | yes |
| `fs-vm-touch-whisking` | yes |
| `fs-whisking-rate` | yes |
| `fs-whisking-phase` | yes |
| `fs-touch-and-whisking` | yes |
| `exc-mainly-touch` | yes |
| `vpm-photoinhibition` | blocked: data missing |
| `exc-thalamus-photoinhibition` | yes |
| `thalamus-drives-fs` | run; blocked, too few cells |
| `rabies-vpm-input` | blocked: data missing |
| `ion-cut-exc-no-touch`, `ion-cut-exc-whisking`, `ion-cut-exc-phase`, `ion-cut-fs-no-touch`, `sensory-input-drives-fs`, `ion-cut-fs-phase`, `ion-cut-behavior` | blocked: metadata insufficient |
| `pv-enphr-suppression` | yes |
| `fs-suppress-movement-spiking` | yes |
| `fs-enhance-touch-selectivity` | yes |
| `exc-touch-spikes-enphr` | yes |
| `auc-weak-strong` | yes |
| `auc-groups` | run; blocked, readings disagree |
| `behavior-false-alarms` | run; blocked, readings disagree |

Readings. The claims file names readings for details the paper leaves open (O1 filter direction, O2
samples of the rank-sum test, O3 threshold rule, O4 resting Vm, O5 light windows, O6 fraction of
control, O7 laser powers, O8 derivative). Each is run where it applies. Cell groups hold the cells
of every mouse ("all"), since the files do not mark the ION-cut mice; a variant without the mice
flagged as lacking touch input ("no_flagged") is shown and does not decide an outcome.

In [ ]:
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import yu2016

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
recs = yu2016.load_all()
vpm = [r for r in recs if r["kind"] == "VPM"]
l4 = [r for r in recs if r["kind"] == "L4"]
by_path = {r["path"]: r for r in l4}
print(f"loaded {len(recs)} files from dandiset {yu2016.DANDISET_ID} version {yu2016.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

No random numbers are drawn in the data check. Each check of the metadata against the values is stated under its output.

In [ ]:
vc = yu2016.versions_check()
print("Versions (identifier, last modified):", vc["versions"])
print("Assets in the pinned version:", vc["pinned_assets"], "; in the draft:", vc["draft_assets"],
      "; same paths and sizes:", vc["draft_same_paths_and_sizes"])
files = pd.DataFrame([{"kind": r["kind"], "subject": r["subject"], "genotype": r["genotype"] or "(none)",
                       "path": r["path"], "trials": len(r["trials"])} for r in recs])
print("Files by kind:", files.kind.value_counts().to_dict(), "; subjects:", files.subject.nunique(),
      "(VPM", files[files.kind == "VPM"].subject.nunique(), ", L4", files[files.kind == "L4"].subject.nunique(),
      ", in both", len(set(files[files.kind == "VPM"].subject) & set(files[files.kind == "L4"].subject)), ")")
print("Trials: VPM", int(files[files.kind == "VPM"].trials.sum()), ", L4", int(files[files.kind == "L4"].trials.sum()))
lines = yu2016.supplement_mouse_lines()
line_of = dict(zip(lines.subject, lines.line))
files["line"] = files.subject.map(line_of)
print(f"\nSupplementary Table 1 lists {len(lines)} mice; {files.subject.drop_duplicates().isin(lines.subject).sum()} "
      f"of the {files.subject.nunique()} mice of the dandiset are in it.")
print("Files by genotype in the files (rows) against the mouse line in Supplementary Table 1 (columns):")
print(pd.crosstab([files.kind, files.genotype], files.line.fillna("(not listed)")).to_string())
print("Mice without a genotype in their files, or given as Ai32:",
      files[files.genotype.isin(["(none)", "Ai32"])].drop_duplicates("subject").groupby(["genotype", "line"]).subject.apply(list).to_dict())

meta = pd.DataFrame([{"path": r["path"], "subject": r["subject"], "general_fields": ",".join(r["general_fields"]),
                      "subject_fields": ",".join(r["subject_fields"]),
                      "region": r["ic_location"].split(";")[0].replace("brain_region:", "").strip(),
                      "layer": r["ic_location"].split("cortical_layer:")[1].split(";")[0].strip(),
                      "dv_mm": float(r["ic_location"].split("coordinate_dv:")[1].split(";")[0])} for r in l4])
print("\nFields under general/ in the L4 files:", meta.general_fields.value_counts().to_dict())
print("Fields under general/subject:", meta.subject_fields.value_counts().to_dict())
print("Electrode location of the L4 files: region", meta.region.value_counts().to_dict(), "; cortical layer",
      meta.layer.value_counts().to_dict(), f"; depth (coordinate_dv) {meta.dv_mm.min():.2f} to {meta.dv_mm.max():.2f} mm")
print("VPM electrode groups:", pd.Series([loc.split(";")[1].strip() for r in vpm for loc in r["location"]]).value_counts().to_dict())
terms = {"ION": r"\bION\b", "infraorbital": r"(?i)infraorbital", "nerve": r"(?i)\bnerves?\b", "transect": r"(?i)transect",
         "lesion": r"(?i)lesion", "cut": r"(?i)\bcut\b", "surgery": r"(?i)surg", "sham": r"(?i)\bsham\b",
         "control": r"(?i)\bcontrol\b", "rabies": r"(?i)rabies", "histology": r"(?i)histolog", "eNpHR": r"(?i)NpHR|halorhodopsin"}
print("L4 files whose metadata text (every string under general/ and the session description) contains each term:",
      {k: sum(bool(re.search(v, r["metadata_text"])) for r in l4) for k, v in terms.items()})
print("Units in the VPM files:", sum(len(r["units"]) for r in vpm), "in", len(vpm), "files; unit descriptions start with",
      pd.Series([u["desc"].split(".")[0] for r in vpm for u in r["units"]]).value_counts().to_dict(),
      "; VPM files with a photostimulation series:", sum(r["has_light"] for r in vpm))
print("L4 files with an electrode outside barrel cortex:", int((meta.region != "barrel cortex").sum()))

0.220126.1853 is the only published version, and the draft holds the same asset paths and sizes.
The 18 VPM files come from 6 mice and the 130 L4 files from 49 others; all 55 mice are in
Supplementary Table 1. Where a file gives a genotype it agrees with the table's mouse line, but 9 mice
have none (the 6 VPM mice and 3 L4 mice) and the 5 SOM-IRES-Cre x Ai32 mice are given as `Ai32`
(earlier finding 3, still present). The files hold no surgery, lesion, opsin or notes field, and no
text names ION, infraorbital, nerve, transection, cut, sham or eNpHR (earlier findings 2 and 5, still
present). Every L4 file gives barrel cortex, layer 4, at 0.42 to 0.62 mm (earlier finding 4, still
present). The VPM files hold no photostimulation series, and no L4 file is outside barrel cortex, so
the thalamic multiunit recordings of Fig. 6b,c are not in the dandiset.

In [ ]:
mode = pd.DataFrame([{"path": r["path"], "subject": r["subject"], "line": line_of[r["subject"]], "label": r["label_mode"],
                      "from_trace": r["mode"], "vm_median": r["vm_median"], "lowfreq_sd": r["lowfreq_sd_mv"], "vm_hz": r["vm_hz"],
                      "trial_s": r["vm_trial_s"], "spikes": len(r["spikes"]),
                      "cur_conv": r.get("current_conversion", np.nan), "cur_unit": r.get("current_unit", ""),
                      "cur_test": r.get("current_end_of_trial_median", np.nan), "cur_min": r.get("current_trial_min", np.nan),
                      "trials_near_0mV": int(np.sum(r["trial_vm_median"] > -20)) if r["mode"] == "whole-cell" else 0}
                     for r in l4])
print("Recording mode in the session description (rows) against the trace (columns; whole-cell when the median trace "
      "is below -20 mV):")
print(pd.crosstab([mode.line, mode.label], mode.from_trace).to_string())
print("\nMedian of the trace and s.d. of its 1 to 100 Hz band (stored values; unit 'volts', conversion 0.001, so mV):")
print(mode.groupby("from_trace")[["vm_median", "lowfreq_sd"]].describe().round(2).T.to_string())
print("Trace sampling rate (Hz) and trial length (s), files:", mode.groupby(["vm_hz", "trial_s"]).size().to_dict())
print("First file at 20 kHz:", min(p.split("_ses-")[1][:8] for p in mode[mode.vm_hz == 20000].path),
      "; last at 10 kHz:", max(p.split("_ses-")[1][:8] for p in mode[mode.vm_hz == 10000].path))
ts = pd.DataFrame([{"outside": r["vm_outside_trials"], "first_ms": r["vm_block_first_rel"] * 1000} for r in l4])
print("Trace samples outside every trial:", int(ts.outside.sum()), "; largest offset of a trial's first sample from the "
      "trial start (ms):", round(ts.first_ms.max(), 2))
print("\nInjected current (CurrentClampStimulus): unit", mode.cur_unit.unique().tolist(), ", conversion",
      mode.cur_conv.unique().tolist(), "; stored value of the end-of-trial test pulse in whole-cell files: median",
      round(mode[mode.from_trace == "whole-cell"].cur_min.median(), 3), "(files at about -0.1:",
      int(np.isclose(mode[mode.from_trace == "whole-cell"].cur_min, -0.1, atol=0.01).sum()), "of",
      int((mode.from_trace == "whole-cell").sum()), "); the Methods give a -0.1 nA test pulse")
print("\nWhole-cell files with trials whose median Vm is above -20 mV (a cell held near 0 mV by current):")
print(mode[mode.trials_near_0mV > 0][["path", "line", "trials_near_0mV", "vm_median"]].to_string(index=False))
print("Cell-attached files (by the trace) with a 1 to 100 Hz s.d. above 0.5 mV:",
      mode[(mode.from_trace == "cell-attached") & (mode.lowfreq_sd > 0.5)][["path", "vm_median", "lowfreq_sd"]].round(2).to_dict("records"))

The recording mode in the session description disagrees with the trace for 24 files, all from
PV-IRES-Cre mice: 23 described as whole-cell sit near 0 mV with a 1 to 100 Hz s.d. of 0.08 to 0.3 mV
and small extracellular spikes, as cell-attached recordings do, and one described as juxtacellular
sits near -65 mV (earlier finding 6, still present). The mode read from the trace is used below; the
stored trace is in mV (conversion 0.001 to volts), which the whole-cell resting potentials confirm.
The trace is sampled at 10 kHz up to March 2015 and at 20 kHz from late August 2015, which only the
timestamps state (earlier finding 8, still present); timestamps start at the trial start and no
sample lies outside a trial.

New: the injected current (`CurrentClampStimulus`) is stored with unit amperes and conversion 1e-6,
so a stored -0.1 would be -0.1 uA. The end-of-trial test pulse, which the Methods give as -0.1 nA,
is stored as about -0.1, so the stored values are in nA and the conversion is wrong by a factor of
1,000. Only one whole-cell file has trials held near 0 mV (13 trials of the PV-ChR2 cell), so the
recordings of L4 excitatory neurons depolarized to reveal inhibition (Fig. 3, 6 neurons) are not in
the dandiset. Two cell-attached files have a larger slow component (1.1 and 1.3 mV), still far below
the whole-cell values.

In [ ]:
wh = []
for r in recs:
    b = r["beh"]
    row = {"kind": r["kind"], "path": r["path"], "principal": b["principal"] is not None, "whiskers": len(b["whisker_names"])}
    if b["principal"] is not None:
        row.update(dt_ms=b["dt_median"] * 1000, angle_min=b["angle_range"][0], angle_max=b["angle_range"][1],
                   angle_nan=b["angle_nan"], touches=len(b["touch_on"]), licks=len(b["licks"]),
                   samples_per_trial=float(np.median(np.bincount(b["trial"][b["trial"] >= 0])[np.bincount(b["trial"][b["trial"] >= 0]) > 0])))
    wh.append(row)
wh = pd.DataFrame(wh)
p = wh[wh.principal]
print("Files with a whisker named principal:", len(p), "of", len(wh), "; without:", [x.split("/")[1] for x in wh[~wh.principal].path])
print("Whisker sampling interval (ms):", sorted(p.dt_ms.round(3).unique().tolist()), "; samples per trial, files by value:",
      p.samples_per_trial.value_counts().sort_index().to_dict())
print("Principal whisker angle (deg): min", round(p.angle_min.min(), 1), ", max", round(p.angle_max.max(), 1),
      "; median fraction of NaN samples", round(p.angle_nan.median(), 4))
print("Touch onsets of the principal whisker per file: median", int(p.touches.median()), ", range", int(p.touches.min()),
      "to", int(p.touches.max()), "; lick onsets per file: median", int(p.licks.median()))
ep = pd.DataFrame([{"kind": r["kind"], **{f"{f}:{k}": r["beh"][f][k].sum() / 1000 for f in yu2016.FILTERS for k in ("whisk", "nonwhisk")},
                    **{f"{f}:onsets": len(r["beh"][f]["onsets"]) for f in yu2016.FILTERS}} for r in recs if r["beh"]["principal"]])
print("\nWhisking epochs and nonwhisking periods (s per file) and whisking onsets per file, medians, by filter reading:")
print(ep.groupby("kind").median().round(1).T.to_string())

rows = []
for r in recs:
    b = r["beh"]
    if b["principal"] is None:
        continue
    on_t = b["t"][b["touch_on"]]
    tr = r["trials"]
    for s, e, ft in zip(tr.start, tr.stop, tr.first_touch_time):
        x = on_t[(on_t >= s) & (on_t < e)]
        if len(x) and np.isfinite(ft):
            rows.append({"kind": r["kind"], "table_rel": ft - s, "series_rel": x[0] - s})
ft = pd.DataFrame(rows)
for kind, g in ft.groupby("kind"):
    print(f"\n{kind}: {len(g)} trials with a touch. first_touch_time minus trial start, median {g.table_rel.median():.4f} s; "
          f"first touch onset of the series, median {g.series_rel.median():.3f} s; table equals series within 2 ms: "
          f"{np.mean(np.abs(g.table_rel - g.series_rel) < 0.002):.3f}; table x 1000 within 2 ms of the series: "
          f"{np.mean(np.abs(g.table_rel * 1000 - g.series_rel) < 0.002):.3f}")

# VPM: spikes per touch (VPM definition) of each unit for the touches of each tracked whisker
rows = []
for r in vpm:
    b = r["beh"]
    allm = np.ones(len(r["trials"]), bool)
    for u in r["units"]:
        c = yu2016.counts_1k(u["spikes"], b)
        res = {}
        for name, w in b["whisker_touches"].items():
            bb = yu2016.with_whisker(b, name)
            s = yu2016.spikes_per_touch(c, b["trial"], yu2016.touches(bb, allm, 0.025), "vpm")
            res[name] = s["spt"] if s["n_touches"] >= 20 else np.nan
        pw = b["principal"].split("_")[-1].upper()
        best = max(res, key=lambda k: -np.inf if np.isnan(res[k]) else res[k])
        rows.append({"path": r["path"], "unit": u["unit"], "file_pw": pw, "best": best, "spt_pw": res[pw], "spt_best": res[best],
                     "whiskers": len(res)})
pwu = pd.DataFrame(rows)
multi = pwu[pwu.whiskers > 1]
print(f"\nVPM units in files with more than one tracked whisker: {len(multi)}; units whose largest response (spikes per touch, "
      f"at least 20 touches) is to a whisker other than the file's principal one: {int((multi.best != multi.file_pw).sum())}")
print(multi[multi.best != multi.file_pw].assign(path=lambda d: d.path.str.split("/").str[1].str[:30]).round(2).to_string(index=False))

The whisker variables are at 1 kHz, 5,000 samples (5 s) per trial, in degrees with a plausible
range. 8 files name no principal whisker (earlier finding 9, still present); they are left out of
every measure that needs touch or whisking. In the L4 files `first_touch_time` matches the touch
series only when multiplied by 1000 (0.956 of trials), so the latency was divided by 1000 once too
often; in the VPM files it is correct (earlier finding 7, still present). The analysis uses the touch
series. In the VPM files each file names one principal whisker, though the session description says
each unit has its own: 11 of the 20 units in files with more than one tracked whisker respond most to
another whisker, some strongly (earlier finding 9, now shown per unit).

In [ ]:
light = []
for r in l4:
    tr = r["trials"]
    for site, L in r["light"].items():
        w = yu2016.light_windows(r["light"], site, tr)
        modes = tr.photo_stim_mode.to_numpy()[w.trial.to_numpy(int)] if len(w) else []
        dur = np.diff(L["pulses"], axis=1).ravel() if len(L["pulses"]) else np.zeros(0)
        light.append({"line": line_of[r["subject"]], "site_region": L["region"], "lambda_nm": L["lambda_nm"],
                      "power_max": L["power_max"], "lit_trials": len(w), "short_pulses": int((dur < 0.0075).sum()),
                      "long_epochs": int((dur > 0.1).sum()), "isolated_pulses": len(yu2016.isolated_pulses(r["light"], site)),
                      "pulses_per_lit_trial": w.n_pulses.median() if len(w) else 0, "modes": ",".join(sorted(set(modes)))})
light = pd.DataFrame(light)
print("Photostimulation sites by mouse line, labelled region and wavelength:")
print(light.groupby(["line", "site_region", "lambda_nm"]).agg(
    files=("lit_trials", "size"), with_light=("lit_trials", lambda x: int((x > 0).sum())), lit_trials=("lit_trials", "sum"),
    short_pulses=("short_pulses", "sum"), isolated=("isolated_pulses", "sum"), long_epochs=("long_epochs", "sum"),
    pulses_per_trial=("pulses_per_lit_trial", "median"), power_max=("power_max", "max")).to_string())
print("\nphoto_stim_mode values of the lit trials, by site region:")
print(light[light.lit_trials > 0].groupby("site_region").modes.value_counts().to_string())
desc = pd.DataFrame([r["trials"].attrs.get("descriptions", {}) for r in recs])
print("Descriptions of trial-table columns (distinct values over files):",
      {c: sorted(desc[c].dropna().unique().tolist()) for c in ["type", "response", "photo_stim_mode", "photo_stim_power", "stim_present"] if c in desc})
print("Trial table photo_stim_power is NaN in", sum(int(r["trials"].photo_stim_power.isna().all()) for r in l4), "of", len(l4), "L4 files")
# the pulse rate of the trains at the site labelled VPM
rates = []
for r in l4:
    for site, L in r["light"].items():
        if L["region"] == "VPM" and len(L["pulses"]) > 10:
            d = np.diff(L["pulses"][:, 0])
            rates.append(1 / np.median(d[d < 0.05]) if np.any(d < 0.05) else np.nan)
print("Pulse rate within trains at the site labelled VPM (Hz), files:", pd.Series(np.round(rates)).value_counts().to_dict())

`photo_stim_mode` and `type` have empty descriptions, `photo_stim_power` is NaN in every L4 file,
and the mode labels do not map to one kind of light (earlier finding 5, still present). The light
epochs are read from the photostimulation series. In VGAT-ChR2 mice the site labelled VPM is given as
594 nm, and pulses at the site labelled vS1 (594 nm) drive VGAT-ChR2 cells at short latency (below);
ChR2 is driven by blue light, so these wavelengths look wrong (earlier finding 10, still present).
New: the trains at the site labelled VPM run at 100 Hz in 21 files and at 50 Hz in 12, while the
Methods give 100 Hz. Trial types are spelled both `Nogo` and `NoGo` (new); they are read as one label.

In [ ]:
# Alignment of spikes and behaviour: each cell's touch response (spikes per touch, VPM or Fig. 5 definition) and
# phase-coupling r2 with its spikes in their own trial, and moved to the next trial at the same time after its start.
rows = []
for r in recs:
    b = r["beh"]
    if b["principal"] is None:
        continue
    allm = np.ones(len(r["trials"]), bool) if r["kind"] == "VPM" else ~yu2016.lit_trials(r["light"], r["trials"])
    cells = [(u["unit"], u["spikes"]) for u in r["units"]] if r["kind"] == "VPM" else [(0, r["spikes"])]
    for unit, sp in cells:
        if len(sp) < 50:
            continue
        out = {"kind": r["kind"], "path": r["path"], "unit": unit}
        for tag, s in (("true", sp), ("shift", yu2016.shift_spikes(sp, r["trials"]))):
            c = yu2016.counts_1k(s, b)
            c = np.where(allm[np.clip(b["trial"], 0, None)], c, 0)
            out[f"spt_{tag}"] = yu2016.spikes_per_touch(c, b["trial"], yu2016.touches(b, allm), "vpm" if r["kind"] == "VPM" else "cortex5")["spt"]
            vals, _ = yu2016.phase_binned(b, "filtfilt", count=c)
            out[f"r2_{tag}"] = yu2016.phase_fit(vals)["r2"]
        rows.append(out)
align = pd.DataFrame(rows)
for kind, g in align.groupby("kind"):
    g = g.dropna()
    print(f"{kind}: {len(g)} units or cells with at least 50 spikes. Spikes per touch, median own trial {g.spt_true.median():.3f}, "
          f"next trial {g.spt_shift.median():.3f}, larger in own trial for {(g.spt_true > g.spt_shift).sum()} "
          f"(signed-rank p = {stats.wilcoxon(g.spt_true, g.spt_shift).pvalue:.2g}); phase r2, median own {g.r2_true.median():.2f}, "
          f"next {g.r2_shift.median():.2f} (p = {stats.wilcoxon(g.r2_true, g.r2_shift).pvalue:.2g}); r2 > 0.25 own "
          f"{(g.r2_true > 0.25).mean():.2f}, next {(g.r2_shift > 0.25).mean():.2f}")
bad = align[(align.spt_true <= align.spt_shift) & (align.r2_true <= align.r2_shift)]
print("Units or cells where neither measure is larger in their own trial:", len(bad), "of", len(align))

# Duplicates: identical traces, identical spike trains, or identical trial tables across files
hashes = pd.Series([r["vm_hash"] for r in l4])
print("\nL4 files sharing a trace hash with another file:", int(hashes.duplicated(keep=False).sum()))
sk = {}
for r in vpm:
    for u in r["units"]:
        sk.setdefault((len(u["spikes"]), round(float(u["spikes"][:5].sum()), 4)), []).append((r["path"], u["unit"]))
print("VPM units with identical spike count and first spike times:", [v for v in sk.values() if len(v) > 1])
tt = pd.Series([(r["subject"], len(r["trials"]), round(float(r["trials"].start.iloc[-1]), 3)) for r in recs])
print("Files sharing subject, trial count and last trial start with another file:", int(tt.duplicated(keep=False).sum()))

Spikes and behaviour share a clock in the data as a whole: for VPM units and L4 cells, the touch
response and the phase coupling are larger with spikes in their own trial than moved to the next
trial (signed-rank p printed above). This cannot be shown for a cell that responds to neither; 9 of
121 such cells are consistent with the clock, not shown to share it. No two files share a trace, a
VPM spike train or a trial table, so no recording is deposited twice. Chronic VPM units recorded on
several days may be the same neurons; the files give no identifiers that link them, and each unit is
counted once, as in the paper's 29.

In [ ]:
# Cell classes, by the rules of the claims file's header (see the helper's docstring for the tests):
# FS: in VGAT-ChR2-EYFP mice, driven by 5 ms light pulses at the cortical site within 5 ms of onset (isolated pulses,
#     one-sided binomial p < 0.001; reading p < 0.01 shown), and, as a reading, inside the Supp. Fig. 6 waveform
#     boundary (cell-attached cells only); in PV-IRES-Cre mice, PV cells inactivated by the eNpHR light (rate inside
#     the light lower than in the same span of trials without light, Mann-Whitney over trials, p < 0.05); in lines
#     without a tag, whole-cell cells with spikes no wider than 0.35 ms at half height (the largest FS width the Methods
#     give). Every other L4 cell is excitatory (E).
def light_effect(r):
    """Spike rate inside the light (median span of the lit trials) against the same span of trials without light."""
    out = None
    ctl = ~yu2016.lit_trials(r["light"], r["trials"])
    for site, L in r["light"].items():
        w = yu2016.light_windows(r["light"], site, r["trials"])
        if len(w) == 0 or not (np.diff(L["pulses"], axis=1) > 0.1).any():
            continue
        lit = np.zeros(len(r["trials"]), bool)
        lit[w.trial.to_numpy(int)] = True
        win = (float(np.median(w.rel_start)), float(np.median(w.rel_stop)))
        sp, st = np.sort(r["spikes"]), r["trials"].start.to_numpy()
        per = lambda m: (np.searchsorted(sp, st[m] + win[1]) - np.searchsorted(sp, st[m] + win[0])) / (win[1] - win[0])
        a, b = per(lit), per(ctl)
        p = stats.mannwhitneyu(a, b, alternative="less").pvalue if len(a) and len(b) and (a.sum() + b.sum()) > 0 else np.nan
        out = {"light_site": L["region"], "light_win": win, "lit_trials": int(lit.sum()), "ctl_trials": int(ctl.sum()),
               "rate_ctl": b.mean(), "rate_light": a.mean(), "light_p_lower": p}
    return out

rows = []
for r in l4:
    row = {"path": r["path"], "subject": r["subject"], "line": line_of[r["subject"]], "mode": r["mode"], "spikes": len(r["spikes"]),
           "fwhm_ms": r["fwhm_ms"], "ca_duration_ms": r["ca_duration_ms"], "ca_ratio": r["ca_ratio"], "pw": r["beh"]["principal"] is not None}
    best = None
    for site, L in r["light"].items():
        if "vS1" not in L["region"]:
            continue
        lr = yu2016.light_pulse_response(r["light"], r["spikes"], site)
        if lr["n_pulses"] >= yu2016.TAG_MIN_PULSES and (best is None or lr["p_evoked"] - lr["p_base"] > best["p_evoked"] - best["p_base"]):
            best = {**lr, "tag_site": L["region"]}
    if best:
        row.update(tag_site=best["tag_site"], pulses=best["n_pulses"], p_evoked=best["p_evoked"], p_base=best["p_base"],
                   tag_p=best["p_value"], tag_latency_ms=best["median_latency_ms"])
    le = light_effect(r) if row["line"] == "PV-IRES-Cre" else None
    if le:
        row.update(le)
    rows.append(row)
cells = pd.DataFrame(rows)
cells["tag_tested"] = cells.get("pulses", pd.Series(np.nan, index=cells.index)).notna()
cells["driven"] = cells.tag_p < yu2016.TAG_MAX_P
cells["driven_01"] = cells.tag_p < 0.01
cells["in_boundary"] = (cells.ca_duration_ms < yu2016.FS_MAX_DURATION_MS) & (cells.ca_ratio < yu2016.FS_MAX_RATIO)
cells["pv_inactivated"] = (cells.line == "PV-IRES-Cre") & (cells.light_p_lower < 0.05) & (cells.rate_light < cells.rate_ctl)
untagged_lines = ~cells.line.isin(["VGAT-ChR2-EYFP", "PV-IRES-Cre"])
cells["thin_wc"] = untagged_lines & (cells["mode"] == "whole-cell") & (cells.fwhm_ms <= 0.35)

def classify(driven_col="driven", boundary=True):
    driven = (cells.line == "VGAT-ChR2-EYFP") & cells[driven_col]
    fs = driven & ((cells["mode"] == "whole-cell") | cells.in_boundary) if boundary else driven
    cls = np.where(fs | cells.pv_inactivated | cells.thin_wc, "FS", "E")
    # driven (GABAergic) cells outside the boundary are neither FS nor excitatory (Supp. Fig. 6)
    return np.where(driven & ~fs, "other GABAergic", cls)

cells["cls"] = classify()                        # the claims file's rule: driven (p < 0.001) and inside the boundary
cells["cls_01"] = classify("driven_01")          # reading: driven at p < 0.01
cells["cls_tag"] = classify(boundary=False)      # variant, not the claims file's rule: driven, any waveform
print("L4 cells by mouse line, mode and class (driven at p < 0.001 and, cell-attached, inside the waveform boundary):")
print(pd.crosstab([cells.line, cells["mode"]], cells.cls).to_string())
print("Cells whose class changes with the tag at p < 0.01:", cells[cells.cls != cells.cls_01].path.str.split("/").str[1].str[:40].tolist())
print("Cells that are FS without the waveform boundary (driven, outside it):",
      cells[cells.cls != cells.cls_tag][["path", "ca_duration_ms", "ca_ratio"]].assign(path=lambda x: x.path.str.split("/").str[1].str[:36]).round(2).values.tolist())
vg = cells[(cells.line == "VGAT-ChR2-EYFP")]
print(f"\nVGAT-ChR2 cells: {len(vg)}; tested with isolated cortical pulses {int(vg.tag_tested.sum())}; driven (p < 0.001) "
      f"{int(vg.driven.sum())}, (p < 0.01) {int(vg.driven_01.sum())}")
d = vg[vg.driven]
print("Driven cells: evoked spike probability median", round(d.p_evoked.median(), 2), "(baseline", round(d.p_base.median(), 2),
      "); first-spike latency median", round(d.tag_latency_ms.median(), 2), "ms, range", round(d.tag_latency_ms.min(), 2), "to",
      round(d.tag_latency_ms.max(), 2))
print("VGAT-ChR2 cells without isolated cortical pulses:", vg[~vg.tag_tested][["path", "mode", "spikes", "fwhm_ms"]]
      .assign(path=lambda x: x.path.str.split("/").str[1].str[:40]).round(2).to_dict("records"))
ca = vg[vg["mode"] == "cell-attached"]
print("\nCell-attached VGAT-ChR2 cells, Supp. Fig. 6 measures (peak-to-trough duration ms, peak/trough ratio), by tag:")
print(ca.groupby("driven")[["ca_duration_ms", "ca_ratio"]].describe().round(2).T.to_string())
print("Driven cell-attached cells inside the boundary (duration < 0.27 ms and ratio < 2.8):", int((ca.driven & ca.in_boundary).sum()),
      "of", int(ca.driven.sum()), "; undriven inside it:", int((~ca.driven & ca.in_boundary).sum()), "of", int((~ca.driven).sum()))
pv = cells[cells.line == "PV-IRES-Cre"].dropna(subset=["rate_ctl"])
print(f"\nPV-IRES-Cre cells with the eNpHR light: {len(pv)} (light span {sorted(set(tuple(np.round(w, 2)) for w in pv.light_win))[:3]} ...); "
      f"rate lower with light (p < 0.05): {int(pv.pv_inactivated.sum())}; ratio light/control below 0.5 among them: "
      f"{int((pv.pv_inactivated & (pv.rate_light < 0.5 * pv.rate_ctl)).sum())}")
print(pv[pv.pv_inactivated | (pv.rate_light < 0.8 * pv.rate_ctl)][["path", "mode", "spikes", "rate_ctl", "rate_light", "light_p_lower", "ca_duration_ms", "ca_ratio"]]
      .assign(path=lambda x: x.path.str.split("/").str[1].str[:40]).round(3).to_string(index=False))
print("\nThin-spiking whole-cell cells of lines without a tag:", cells[cells.thin_wc][["path", "line", "fwhm_ms", "spikes"]]
      .assign(path=lambda x: x.path.str.split("/").str[1].str[:40]).round(2).to_dict("records"))
print("Whole-cell spike width at half height (ms) of the lines without a tag:",
      cells[untagged_lines & (cells["mode"] == "whole-cell")][["line", "fwhm_ms"]].round(2).values.tolist())
# Against Supplementary Table 2 (cells per line, mode and type; control and ION-cut cells together)
paper = pd.DataFrame([("VGAT-ChR2-EYFP", "whole-cell", 32, 5), ("VGAT-ChR2-EYFP", "cell-attached", 6, 24),
                      ("PV-IRES-Cre", "whole-cell", 5, 0), ("PV-IRES-Cre", "cell-attached", 38, 4),
                      ("PV-IRES-Cre x Ai32", "whole-cell", 1, 0), ("C57BL/6", "whole-cell", 7, 1), ("PV-ChR2", "whole-cell", 0, 1),
                      ("SOM-IRES-Cre x Ai32", "whole-cell", 1, 0), ("SOM-IRES-Cre x Ai32", "cell-attached", 4, 0)],
                     columns=["line", "mode", "paper_E", "paper_FS"])
ours = cells.groupby(["line", "mode"]).cls.value_counts().unstack(fill_value=0).reindex(columns=["E", "FS", "other GABAergic"], fill_value=0)
cmp_ = paper.merge(ours.reset_index().rename(columns={"E": "files_E", "FS": "files_FS", "other GABAergic": "files_other"}),
                   on=["line", "mode"], how="outer").fillna(0)
print("\nCells per line, mode and type: Supplementary Table 2 (E includes 9 whole-cell ION-cut cells, FS 10 cell-attached ION-cut cells) "
      "against the files:")
print(cmp_.to_string(index=False))
print("Totals: paper E", int(cmp_.paper_E.sum()), "FS", int(cmp_.paper_FS.sum()), "; files E", int(cmp_.files_E.sum()), "FS",
      int(cmp_.files_FS.sum()), "other GABAergic", int(cmp_.files_other.sum()))

No file names the cell type of its L4 neuron (earlier finding 1, still present), so the classes are
built from the data by the claims file's rules. In VGAT-ChR2 mice 24 cells are driven by isolated
cortical pulses within 5 ms (latency 1.6 to 4.6 ms); Supplementary Table 2 counts 29 FS cells there
(19 control and 10 ION-cut). The waveform boundary of Supplementary Fig. 6, as read in the claims
file, keeps 12 of the 20 driven cell-attached cells; the other 8 are GABAergic but outside it and are
left out of both classes. The tag at p < 0.01 adds 3 cells, all outside the boundary, so it changes
no FS group. 29 VGAT-ChR2 cells were never given isolated cortical pulses and count as excitatory. In
PV-IRES-Cre mice 3 cells are inhibited by the light (the paper has 4 PV cells); in the lines without
a tag one whole-cell cell has spikes 0.25 ms wide. The PV-ChR2 cell, FS in Supplementary Table 2, has
spikes 0.44 ms wide here and counts as excitatory. The result: 20 FS cells against the table's 35
(25 control and 10 ION-cut), and 102 excitatory cells against 94.

## Measures for every unit and cell

Each VPM unit and L4 cell with a named principal whisker, over its trials without light, by the definitions of the helper module.

In [ ]:
# Every measure of every VPM unit and L4 cell over its trials without light (claims, Measures; helper docstring).
def ctl_mask(r):
    return np.ones(len(r["trials"]), bool) if r["kind"] == "VPM" else ~yu2016.lit_trials(r["light"], r["trials"])

rows, vrows = [], []
for r in recs:
    b = r["beh"]
    if b["principal"] is None:
        continue
    m = ctl_mask(r)
    if r["kind"] == "VPM":
        for u in r["units"]:
            rows.append({"path": r["path"], "subject": r["subject"], "unit": u["unit"], "cls": "VPM", "mode": "probe",
                         **yu2016.spike_measures(u["spikes"], b, m, "vpm", np.random.default_rng(u["unit"]))})
        continue
    c = cells.set_index("path").loc[r["path"]]
    rows.append({"path": r["path"], "subject": r["subject"], "unit": 0, "cls": c.cls, "mode": r["mode"],
                 **yu2016.spike_measures(r["spikes"], b, m, "cortex5", np.random.default_rng(0), fs_fit=True)})
    if r["mode"] == "whole-cell":
        for filt in yu2016.FILTERS:
            for reading in ("a", "b"):
                v = yu2016.vm_measures(r, m, filt, reading)
                vrows.append({"path": r["path"], "subject": r["subject"], "filt": filt, "O3": reading, "touch_avg_n": r["touch_avg_n"],
                              **{k: x for k, x in v.items() if np.ndim(x) == 0}, "touch_pre": v["touch_pre"], "touch_dvm": v["touch_dvm"],
                              "whisk_pre": v["whisk_pre"], "whisk_dvm": v["whisk_dvm"]})
meas = pd.DataFrame(rows)
vmm = pd.DataFrame(vrows)
for col in ["cls_01", "cls_tag"]:
    meas[col] = meas.path.map(cells.set_index("path")[col]).fillna("VPM")
print("Units and cells measured (named principal whisker):", meas.cls.value_counts().to_dict(), "; whole-cell cells with Vm measures:",
      vmm.path.nunique())
print("Dropped for lack of a named principal whisker:", cells[~cells.pw].cls.value_counts().to_dict())

In [ ]:
# Mice without touch input. ION transection removes the input from the whiskers (Fig. 7); the files do not mark it.
# A cell has a touch input when its spikes per touch (Fig. 5 definition) has a 95% bootstrap interval above zero, or,
# whole-cell, its average touch PSP is at least 1 mV. Mice where no cell has one are flagged.
vm0 = vmm[(vmm.filt == "filtfilt") & (vmm.O3 == "a")].set_index("path")
cm = meas[meas.cls != "VPM"].copy()
cm["line"] = cm.subject.map(line_of)
cm["psp_mv"] = cm.path.map(vm0.amp_mv)
cm["spike_input"] = cm.spt_lo > 0
cm["vm_input"] = cm.psp_mv >= 1.0
perf = {r["path"]: r["trials"].response.isin(["Hit", "CR"]).mean() for r in l4}
cm["perf"] = cm.path.map(perf)
mice = cm.groupby("subject").agg(line=("line", "first"), cells=("path", "size"), whole_cell=("mode", lambda x: int((x == "whole-cell").sum())),
                                 fs=("cls", lambda x: int((x == "FS").sum())), spike_input=("spike_input", "sum"),
                                 vm_input=("vm_input", "sum"), max_psp_mv=("psp_mv", "max"), performance=("perf", "mean"))
mice["no_touch_input"] = (mice.spike_input + mice.vm_input) == 0
# ION-cut mice are VGAT-ChR2-EYFP mice (Supplementary Table 2) and performed at chance (Results); the variant leaves out the
# VGAT-ChR2 mice without touch input or with performance below 0.5. Both rules were chosen after seeing these values.
vgm = mice.line == "VGAT-ChR2-EYFP"
flagged = sorted(mice[vgm & (mice.no_touch_input | (mice.performance < 0.5))].index)
print("Mice without a cell with touch input, and VGAT-ChR2 mice with performance below 0.56:")
print(mice[mice.no_touch_input | (vgm & (mice.performance < 0.56))].round(2).to_string())
fc = cells[cells.subject.isin(flagged)]
print(f"\nFlagged mice (VGAT-ChR2, no touch input or performance below 0.5): {flagged}; their cells by mode and class:", pd.crosstab(fc["mode"], fc.cls).to_dict())
print("Supplementary Table 2: 9 ION-cut VGAT-ChR2 mice with 9 whole-cell E and 10 cell-attached FS cells.")
print("Performance (proportion correct per recording), median: flagged mice", round(cm[cm.subject.isin(flagged)].perf.median(), 2),
      "; other mice", round(cm[~cm.subject.isin(flagged)].perf.median(), 2))

The files do not mark the 9 ION-cut mice. In 3 VGAT-ChR2 mice no cell has a touch input (the 8
whole-cell excitatory cells of anm318530 and anm318532 have average touch PSPs of 0.75 mV or less),
and in anm275231 and anm273341 the mice performed below 0.5 correct. These 4 mice are consistent with
the ION-cut mice of Fig. 7 and are left out in the variant "no_flagged". The rule was chosen after
seeing the values, and it selects by touch responses, so it favours claims about touch; it never
decides an outcome. It finds 8 of the table's 9 whole-cell ION-cut cells and 7 of its 10
cell-attached ones (3 of which are outside the waveform boundary here).

In [ ]:
# The findings of the earlier attempt (000005-yu-2016-01, version 0.220126.1853), each checked again on this version.
pc_ = pd.crosstab([mode.line, mode.label], mode.from_trace)
mislabel = int(((mode.label == "whole-cell") & (mode.from_trace == "cell-attached")).sum() + ((mode.label == "juxtacellular") & (mode.from_trace == "whole-cell")).sum())
status = [
    ("1 No L4 file names the cell type of its neuron", "still present",
     f"fields under general/subject: {meta.subject_fields.unique().tolist()}; no cell-type field or text in any L4 file"),
    ("2 No file marks the ION-transected mice", "still present",
     f"files whose metadata text names ION, infraorbital, nerve, transection, lesion, cut, surgery or sham: "
     f"{sum(any(re.search(terms[k], r['metadata_text']) for k in ['ION', 'infraorbital', 'nerve', 'transect', 'lesion', 'cut', 'surgery', 'sham']) for r in l4)}"),
    ("3 Genotype missing for the 6 VPM mice and 3 L4 mice; SOM-IRES-Cre x Ai32 given as Ai32", "still present",
     f"mice without a genotype: {files[files.genotype == '(none)'].subject.nunique()}; given as Ai32: {files[files.genotype == 'Ai32'].subject.nunique()}"),
    ("4 Every L4 file gives cortical layer 4", "still present", f"layers given: {meta.layer.unique().tolist()}, depth {meta.dv_mm.min():.2f} to {meta.dv_mm.max():.2f} mm"),
    ("5 Optogenetic conditions not described (photo_stim_mode, power NaN, no opsin)", "still present",
     f"photo_stim_power NaN in {sum(int(r['trials'].photo_stim_power.isna().all()) for r in l4)} of {len(l4)} L4 files; files naming eNpHR: "
     f"{sum(bool(re.search(terms['eNpHR'], r['metadata_text'])) for r in l4)}"),
    ("6 Session description gives the wrong recording mode for 24 of 130 L4 files", "still present",
     f"files whose described mode disagrees with the trace: {mislabel}"),
    ("7 L4 first_touch_time is the trial start plus the latency divided by 1000", "still present",
     f"L4 trials where the table value x 1000 matches the touch series: {np.mean(np.abs(ft[ft.kind == 'L4'].table_rel * 1000 - ft[ft.kind == 'L4'].series_rel) < 0.002):.3f}"),
    ("8 L4 trace at 10 or 20 kHz, stated only by the timestamps", "still present", f"files by rate: {mode.vm_hz.value_counts().to_dict()}"),
    ("9 8 files name no principal whisker; VPM files give one per file though units have their own", "still present",
     f"files without one: {int((~wh.principal).sum())}; VPM units responding most to another whisker: {int((multi.best != multi.file_pw).sum())} of {len(multi)}"),
    ("10 VPM-site light given as 594 nm in VGAT-ChR2 mice; vS1 594 nm pulses drive VGAT-ChR2 cells", "still present",
     f"VGAT-ChR2 cells driven by pulses at sites labelled vS1 594 nm: {int(((cells.tag_site == 'vS1') & cells.driven).sum())}"),
    ("11 No analysis code linked by the paper or the dandiset", "still present", "the dandiset links only the paper (related_publications); the claims file found no code"),
]
print(pd.DataFrame(status, columns=["earlier finding", "status", "evidence here"]).to_string(index=False))

Every finding of the earlier attempt is still present on this version, which is the same version
as before: the dandiset has not changed since 2022 (draft and published assets are identical). The
new findings of this attempt are the current conversion, the missing recordings (Fig. 3
depolarized cells, Fig. 6b,c thalamic multiunits, Supplementary Fig. 10 histology), the 50 Hz trains,
and the two spellings of the no-lick trial type.

In [ ]:
# Groups used below. The files do not mark ION-cut mice, so every group holds the cells of all mice ("all"); the
# variant leaves out the VGAT-ChR2 mice flagged as lacking touch input or performing below 0.5 ("no_flagged").
FS_READINGS = {"tag p<0.001, boundary": "cls", "tag p<0.01, boundary": "cls_01"}
FS_VARIANT = {"tag p<0.001, any waveform (variant)": "cls_tag"}

def group(cls, mode=None, sel="all", col="cls"):
    g = meas[meas[col] == cls] if cls != "VPM" else meas[meas.cls == "VPM"]
    if mode:
        g = g[g["mode"] == mode]
    if sel == "no_flagged":
        g = g[~g.subject.isin(flagged)]
    return g

def wilcox(x, y, alternative="two-sided"):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 2 or np.all(x[ok] == y[ok]):
        return np.nan
    return float(stats.wilcoxon(x[ok], y[ok], alternative=alternative).pvalue)

def within(value, expected, tol):
    return bool(np.isfinite(value) and abs(value - expected) <= tol)

## behavior-touches

> Mice touched the pole often when it was within reach (lick trials, 5.7 +- 5.3 touches per trial over 4,147 trials) and rarely when it was distal (0.9 +- 3.4 over 4,173 trials); performance was 0.72 +- 0.11 in cortical and 0.71 +- 0.07 in VPM recordings.

In [ ]:
def touch_counts(r, whiskers="pw", ctl_only=True):
    b, tr = r["beh"], r["trials"]
    if b["principal"] is None:
        return None
    m = ctl_mask(r) if ctl_only else np.ones(len(tr), bool)
    if whiskers == "pw":
        on_t = b["t"][b["touch_on"]]
    else:
        on_t = np.sort(np.concatenate([b["t"][w["on"]] for w in b["whisker_touches"].values()]))
    k = yu2016.per_trial_index(on_t, tr)
    n = np.bincount(k[k >= 0], minlength=len(tr))
    return pd.DataFrame({"type": tr.type.str.capitalize().to_numpy()[m], "touches": n[m], "correct": tr.response.isin(["Hit", "CR"]).to_numpy()[m]})

ctl_cortex = meas[(meas.cls != "VPM") & ~meas.subject.isin(flagged)].path.unique()
res = []
for rec_set, paths in (("a: control cortical recordings", set(ctl_cortex)), ("b: all sessions", {r["path"] for r in recs})):
    for wk in ("pw", "any"):
        for ctl_only in (True, False):
            t = [touch_counts(r, wk, ctl_only) for r in recs if r["path"] in paths]
            t = pd.concat([x for x in t if x is not None])
            go, nogo = t[t.type == "Go"].touches, t[t.type == "Nogo"].touches
            res.append({"recordings": rec_set, "whiskers": wk, "trials": "without light" if ctl_only else "all",
                        "go_trials": len(go), "go_mean": go.mean(), "go_sd": go.std(), "nogo_trials": len(nogo), "nogo_mean": nogo.mean(),
                        "nogo_sd": nogo.std()})
res = pd.DataFrame(res)
print("Trial types in the files (as spelled):", pd.Series([x for r in recs for x in r["trials"].type]).value_counts().to_dict(),
      "; responses:", pd.Series([x for r in recs for x in r["trials"].response]).value_counts().to_dict())
print("Touches per trial (paper: Go 5.7 +- 5.3 over 4,147 trials, tolerance 0.57; Nogo 0.9 +- 3.4 over 4,173, tolerance 0.09):")
print(res.round(2).to_string(index=False))
# performance per recording averaged over cells (cortex) and over units (VPM)
perf_rows = []
for ctl_only in (True, False):
    pc = [(touch_counts(by_path[p], "pw", ctl_only).correct.mean()) for p in ctl_cortex]
    pv_ = [touch_counts(r, "pw", ctl_only).correct.mean() for r in vpm for u in r["units"]]
    perf_rows.append({"trials": "without light" if ctl_only else "all", "cortex_cells": len(pc), "cortex_mean": np.mean(pc), "cortex_sd": np.std(pc, ddof=1),
                      "vpm_units": len(pv_), "vpm_mean": np.mean(pv_), "vpm_sd": np.std(pv_, ddof=1)})
perf_df = pd.DataFrame(perf_rows)
print("\nPerformance (proportion correct per recording), averaged over cortical cells (reading a: control cells, flagged mice left out) "
      "and over VPM units (paper: 0.72 +- 0.11 over 111 cells, 0.71 +- 0.07 over 29 units; tolerances 0.072 and 0.071):")
print(perf_df.round(3).to_string(index=False))
allc = [touch_counts(by_path[p], "pw", True).correct.mean() for p in meas[meas.cls != "VPM"].path.unique()]
print("Over every measured cortical cell, flagged mice included:", len(allc), "cells, mean", round(np.mean(allc), 3))
ok_touch = res[(res.go_mean.sub(5.7).abs() <= 0.57) & (res.nogo_mean.sub(0.9).abs() <= 0.09)]
print("Readings with both touch counts within tolerance:", ok_touch[["recordings", "whiskers", "trials"]].values.tolist())

**Result: partially reproduced.** Under the claims file's reading (b), all sessions and the
principal whisker, mice touched the pole 6.04 times per lick trial (9,794 trials) and 0.85 times per
no-lick trial (8,223 trials), both within tolerance of 5.7 and 0.9. No other reading puts both counts
within tolerance; leaving out the light trials, which the criterion does not ask for, gives 5.74 and
0.80. Performance during the cortical recordings is 0.703 (0.718 without light trials) over 109
cells, within tolerance of 0.72, but during the VPM recordings it is 0.585 over 29 units, outside the
tolerance of 0.71 +- 0.071. The trial counts do not match the paper's 4,147 and 4,173 under any reading.

## vpm-touch

> VPM neurons respond to active touch with a sharp, short-latency increase in spike rate (0.6 +- 0.5 spikes per touch, median 0.4; latency 3.2 +- 1.0 ms, median 2.5 ms).

In [ ]:
v = group("VPM")
print(f"VPM units: {len(v)} (touches with ITI >= 25 ms per unit: median {int(v.n_touches.median())}, range {int(v.n_touches.min())} to {int(v.n_touches.max())})")
print(f"Spikes per touch (VPM definition): mean {v.spt.mean():.3f} (expected 0.6 +- 0.093), median {v.spt.median():.3f} "
      f"(expected 0.4 +- 0.04), s.d. {v.spt.std():.3f}, range {v.spt.min():.2f} to {v.spt.max():.2f}; interval above 0 in {(v.spt_lo > 0).sum()}")
lat = v.dropna(subset=["lat_centre"])
print(f"Latency, units passing the exclusion: {len(lat)} (left out: {v.lat_excluded[v.lat_centre.isna()].value_counts().to_dict()})")
for name in ("lat_edge", "lat_centre"):
    print(f"  {name}: mean {lat[name].mean():.2f} ms (expected 3.2 +- 0.32), median {lat[name].median():.2f} ms (expected 2.5 +- 0.25), "
          f"s.d. {lat[name].std():.2f}, range {lat[name].min():.1f} to {lat[name].max():.1f}")
print("Latencies above 10 ms:", lat[lat.lat_centre > 10][["path", "unit", "lat_centre", "spt"]].assign(path=lambda d: d.path.str[-40:-20]).to_dict("records"))
print("\nBy mouse:")
print(v.groupby("subject").agg(units=("spt", "size"), spt_mean=("spt", "mean"), lat_median=("lat_centre", "median")).round(2).to_string())
bw = pwu.set_index(["path", "unit"])
vv = v.set_index(["path", "unit"]).join(bw[["best", "spt_best"]])
print(f"\nVariant (not the criterion; biased upward): each unit's most responsive tracked whisker, spikes per touch mean "
      f"{vv.spt_best.mean():.3f}, median {vv.spt_best.median():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 2.8))
ax[0].hist(v.spt, bins=np.arange(-0.2, 1.6, 0.1), color="0.5"); ax[0].set_xlabel("spikes per touch"); ax[0].set_ylabel("VPM units")
ax[1].hist(lat.lat_centre, bins=np.arange(0, 41, 1), color="0.5"); ax[1].set_xlabel("latency (ms, bin centre)")
plt.tight_layout(); plt.show()
print("Axes hold every value: spikes per touch", round(v.spt.min(), 2), "to", round(v.spt.max(), 2), "; latency", lat.lat_centre.min(), "to", lat.lat_centre.max())

**Result: not reproduced.** Over the 29 VPM units, spikes per touch average 0.403 (median 0.268),
outside the tolerances of 0.6 and 0.4. The latency median is 2.5 ms with the bin centre (2.0 ms
with its left edge) against 2.5 ms, but the mean is 6.14 ms against 3.2, pulled up by one unit at
71.5 ms; 25 units pass the latency exclusion. The files name one principal whisker per file, and 11
of 20 units in files with several whiskers respond most to another one; taking each unit's most
responsive whisker, a choice biased upward that the criterion does not make, raises the mean to
0.485, still outside the tolerance.

## vpm-whisking

> VPM neurons fire at 5.1 +- 5.5 spikes/s without whisker movement or touch; most (25 of 29) increase their rate during whisking without touch, and the mean rate rises from 5.1 to 13.5 spikes/s. Three of 29 do not increase with whisking but still respond to touch.

In [ ]:
rows = []
for f in yu2016.FILTERS:
    for o2 in ("O2a", "O2b"):
        rows.append({"filter": f, "samples": o2, "increase": int((v[f"{f}:{o2}"] == 1).sum()), "decrease": int((v[f"{f}:{o2}"] == -1).sum()),
                     "nonwhisk_mean": v[f"{f}:rate_nonwhisk"].mean(), "nonwhisk_median": v[f"{f}:rate_nonwhisk"].median(),
                     "nonwhisk_sd": v[f"{f}:rate_nonwhisk"].std(), "whisk_mean": v[f"{f}:rate_whisk"].mean(),
                     "whisk_sd": v[f"{f}:rate_whisk"].std(), "signed_rank_p": "%.2g" % wilcox(v[f"{f}:rate_nonwhisk"], v[f"{f}:rate_whisk"]),
                     "whisk_higher": int((v[f"{f}:rate_whisk"] > v[f"{f}:rate_nonwhisk"]).sum())})
vw = pd.DataFrame(rows)
print("VPM units:", len(v), "; expected: increases 25 +- 2.5 of 29, nonwhisking mean 5.1 +- 1.02, median 3.3 +- 0.33, whisking mean 13.5 +- 2.36 spikes/s")
print(vw.round(4).to_string(index=False))
print("Seconds of nonwhisking and whisking per unit (filtfilt): median", round(v["filtfilt:time_nonwhisk"].median(), 1), "and",
      round(v["filtfilt:time_whisk"].median(), 1))
print("Units without a significant increase (filtfilt, O2a): spikes per touch",
      v[v["filtfilt:O2a"] != 1][["spt", "spt_lo"]].round(2).values.tolist())

**Result: partially reproduced.** The whisking rate is above the nonwhisking rate over the 29
units (signed-rank p = 5.2e-08 and 7.1e-08 for the two filters; higher in 28). 24 or 25 units
increase significantly under three readings (within 25 +- 2.5) and 21 under the forward filter with
per-epoch samples. The mean whisking rate is 11.54 spikes/s with the zero-phase filter (within 13.5 +-
2.36) and 11.01 with the forward filter (outside), and the nonwhisking rate is lower than the paper's
under both (mean 3.76 and 3.93 against 5.1 +- 1.02; median 1.71 and 1.72 against 3.3 +- 0.33).

## vpm-whisking-phase

> The spike rate of VPM neurons is modulated by the phase of whisker movement.

In [ ]:
for f in yu2016.FILTERS:
    print(f"{f}: VPM units with r2 > 0.25: {int((v[f'{f}:r2'] > 0.25).sum())} of {len(v)}; r2 median {v[f'{f}:r2'].median():.2f}; "
          f"coupling strength of coupled units median {v[v[f'{f}:r2'] > 0.25][f'{f}:strength'].median():.2f} spikes/s "
          f"(range {v[v[f'{f}:r2'] > 0.25][f'{f}:strength'].min():.2f} to {v[v[f'{f}:r2'] > 0.25][f'{f}:strength'].max():.2f})")
nv = align[align.kind == "VPM"]
print(f"Reference (not part of the criterion): with spikes moved to the next trial, {int((nv.r2_shift > 0.25).sum())} of {len(nv)} units "
      "have r2 > 0.25 (filtfilt)")
print("By mouse (filtfilt):", v.assign(c=v["filtfilt:r2"] > 0.25).groupby("subject").c.agg(["sum", "size"]).to_dict("index"))

**Result: reproduced.** 25 (zero-phase filter) and 27 (forward filter) of the 29 VPM units have
a sinusoid fit of rate against phase with r2 > 0.25, more than half under both readings. The measure
is weak as a test: with spikes moved to the next trial, 16 of 29 units also pass, which the
criterion does not ask for.

## exc-vm-touch-whisking

> In L4 excitatory neurons (37 whole-cell), resting Vm spans -76 to -45 mV (-63 +- 8 mV); every neuron depolarizes after touch (latency 4.8 +- 0.8 ms, amplitude 6.0 +- 2.9 mV), while whisking causes only a small depolarization or hyperpolarization (1.2 +- 2.1 mV).

In [ ]:
def vm_table(cls, sel, col="cls", filt="filtfilt", o3="a"):
    paths = group(cls, "whole-cell", sel, col).path
    return vmm[vmm.path.isin(paths) & (vmm.filt == filt) & (vmm.O3 == o3)].set_index("path")

for sel in ("all", "no_flagged"):
    e = vm_table("E", sel)
    pos = e[e.amp_mv > 0]
    print(f"[{sel}] whole-cell E cells: {len(e)} (paper 37); average touch PSP from a median of {int(e.touch_avg_n.median())} touches "
          f"(range {int(e.touch_avg_n.min())} to {int(e.touch_avg_n.max())}); amplitude > 0 in {len(pos)}, latency found in "
          f"{int(pos.latency_ms.notna().sum())} of them")
    print(f"   touch latency (expected mean 4.8 +- 0.48 ms): full-rate derivative mean {e.latency_ms.mean():.2f} (s.d. {e.latency_ms.std():.2f}, "
          f"median {e.latency_ms.median():.2f}, range {e.latency_ms.min():.2f} to {e.latency_ms.max():.2f}); 1 ms bins, left edge mean "
          f"{e.latency_1ms_edge.mean():.2f}, centre mean {e.latency_1ms_centre.mean():.2f} (median {e.latency_1ms_centre.median():.2f}, "
          f"max {e.latency_1ms_centre.max():.1f}); amplitude mean {e.amp_mv.mean():.2f} mV (expected 6.0 +- 0.6; s.d. {e.amp_mv.std():.2f}, "
          f"median {e.amp_mv.median():.2f}, range {e.amp_mv.min():.2f} to {e.amp_mv.max():.2f})")
    print(f"   resting Vm O4a mean {e.rest_mean.mean():.2f} mV (s.d. {e.rest_mean.std():.2f}, range {e.rest_mean.min():.1f} to "
          f"{e.rest_mean.max():.1f}); O4b (median per cell) mean {e.rest_median.mean():.2f} (expected -63 +- 6.3)")
    for filt in yu2016.FILTERS:
        ef = vm_table("E", sel, filt=filt)
        print(f"   {filt}: whisking change (a) onset windows mean {ef.whisk_onset_change.mean():.2f} mV (s.d. {ef.whisk_onset_change.std():.2f}; "
              f"cells with onsets {int(ef.whisk_onset_change.notna().sum())}, onsets per cell median {int(ef.n_onsets.median())}); "
              f"(b) epochs mean {ef.whisk_epoch_change.mean():.2f} mV (s.d. {ef.whisk_epoch_change.std():.2f}); expected 1.2 +- 0.35")
e = vm_table("E", "all")
print("Cells without a positive touch PSP or a latency:", e[(e.amp_mv <= 0) | e.latency_ms.isna()][["subject", "amp_mv", "latency_ms", "touch_avg_n"]]
      .round(2).reset_index(drop=True).to_dict("records"))
print("By mouse line:", e.assign(line=e.subject.map(line_of)).groupby("line")[["amp_mv", "latency_ms", "rest_mean"]].mean().round(2).to_dict("index"))

**Result: partially reproduced.** Every one of the 48 whole-cell excitatory cells with enough
touches depolarizes after touch and has a latency (3 cells have fewer than 5 touches). Resting Vm
(mean -61.61 mV, or -61.88 with the median per cell) and the whisking change (0.86 mV with the
onset windows and 1.09 mV with the epochs, zero-phase filter) are within tolerance, but the touch
amplitude (4.77 mV against 6.0 +- 0.6) and the latency (mean 7.68 ms at the full sampling rate
against 4.8 +- 0.48) are not. The group holds 51 cells against the paper's 37 and includes the 8
cells of the flagged mice; without them the amplitude is 5.53 mV and the latency 4.97 ms, both
within tolerance, but that variant does not decide.

## exc-threshold-touch-vs-whisking

> Touch moves the membrane potential of L4 excitatory neurons much closer to spike threshold than whisking does (shift 6.1 +- 3.2 mV against 1.1 +- 2.0 mV).

In [ ]:
rows = []
for sel in ("all", "no_flagged"):
    for filt in yu2016.FILTERS:
        for o3 in ("a", "b"):
            e = vm_table("E", sel, filt=filt, o3=o3).dropna(subset=["shift_whisk", "shift_touch"])
            rows.append({"cells": sel, "filter": filt, "O3": o3, "n": len(e), "shift_whisk_mean": e.shift_whisk.mean(),
                         "shift_whisk_sd": e.shift_whisk.std(), "shift_touch_mean": e.shift_touch.mean(), "shift_touch_sd": e.shift_touch.std(),
                         "touch_larger": int((e.shift_touch > e.shift_whisk).sum()),
                         "p_touch_gt_whisk": "%.2g" % wilcox(e.shift_touch, e.shift_whisk, "greater"),
                         "threshold_median_mv": e.threshold_median.median(), "silent_cells": int((e.threshold_median == -42).sum())})
print("Shift toward threshold (10th percentile of distance to threshold, nonwhisking minus whisking or touch epochs); "
      "expected whisking 1.1 +- 0.33 mV, touch 6.1 +- 0.61 mV, touch > whisking by signed-rank p < 0.05")
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: partially reproduced.** Touch moves the 50 excitatory cells closer to threshold than
whisking does in 43 of them (signed-rank p = 1.9e-09 and 1.3e-10 for the two filters; the two
threshold readings give the same values). The touch shift averages 4.36 to 4.42 mV, outside the
tolerance of 6.1 +- 0.61, and the whisking shift 0.74 mV (zero-phase, outside 1.1 +- 0.33) or 0.90 mV
(forward, within). 17 of the 50 cells fire no spike, so their threshold is the default -42 mV.

## exc-effective-reversal

> The effective reversal potential of the touch response of L4 excitatory neurons is more depolarized than resting Vm, while that of whisking is close to rest and more hyperpolarized than that of touch.

In [ ]:
rows = []
for sel in ("all", "no_flagged"):
    for filt in yu2016.FILTERS:
        e = vm_table("E", sel, filt=filt)
        rt = e.apply(lambda x: yu2016.reversal(x.touch_pre, x.touch_dvm)[0], axis=1)
        rw = e.apply(lambda x: yu2016.reversal(x.whisk_pre, x.whisk_dvm)[0], axis=1)
        ok = rt.notna() & rw.notna()
        pt = yu2016.reversal(np.concatenate(e.touch_pre.to_list()), np.concatenate(e.touch_dvm.to_list()))
        pw_ = yu2016.reversal(np.concatenate(e.whisk_pre.to_list()), np.concatenate(e.whisk_dvm.to_list()))
        rows.append({"cells": sel, "filter": filt, "n_both": int(ok.sum()), "touch_rev_median": rt[ok].median(), "whisk_rev_median": rw[ok].median(),
                     "touch_above_whisk": int((rt[ok] > rw[ok]).sum()), "rest_median": e.rest_mean.median(),
                     "pooled_touch_rev": pt[0], "pooled_touch_slope": pt[1], "pooled_whisk_rev": pw_[0], "pooled_whisk_slope": pw_[1]})
print("Effective reversal potentials (mV): (a) per neuron, medians over neurons with both; (b) one pooled fit per event type")
print(pd.DataFrame(rows).round(2).to_string(index=False))
e = vm_table("E", "all")
print("Per-neuron touch events median", int(e.touch_dvm.map(len).median()), ", whisking onsets median", int(e.whisk_dvm.map(len).median()))

**Result: reproduced.** Fitted per neuron, the touch reversal (median -50.0 mV) lies above the
whisking reversal (median -58.5 or -59.8 mV) in 39 or 38 of 48 neurons and above the median resting
Vm (-63.9 mV); one pooled fit per event type gives -31.25 mV for touch and -54.98 or -56.05 mV for
whisking, both above rest as well.

## fs-touch-response

> L4 FS interneurons fire at 9.3 +- 8.5 spikes/s without whisker movement and fire 1.9 +- 1.0 spikes per touch, with a spike latency of 5.3 +- 2.1 ms.

In [ ]:
rows = []
for rname, col in {**FS_READINGS, **FS_VARIANT}.items():
    for sel in ("all", "no_flagged"):
        g = group("FS", None, sel, col)
        lat = g.dropna(subset=["lat_centre"])
        rows.append({"FS reading": rname, "cells": sel, "n": len(g), "nonwhisk_mean": g["filtfilt:rate_nonwhisk"].mean(),
                     "nonwhisk_mean_lfilter": g["lfilter:rate_nonwhisk"].mean(), "nonwhisk_sd": g["filtfilt:rate_nonwhisk"].std(),
                     "spt_mean": g.spt.mean(), "spt_sd": g.spt.std(), "spt_median": g.spt.median(), "lat_n": len(lat),
                     "lat_edge_mean": lat.lat_edge.mean(), "lat_centre_mean": lat.lat_centre.mean(), "lat_centre_median": lat.lat_centre.median(),
                     "lat_sd": lat.lat_centre.std(), "lat_max": lat.lat_centre.max()})
fsr = pd.DataFrame(rows)
print("FS cells, trials without light; expected nonwhisking 9.3 +- 1.67 spikes/s, spikes per touch 1.9 +- 0.2, latency 5.3 +- 0.53 ms (24 cells)")
print(fsr.round(2).to_string(index=False))
g = group("FS")
print("Main group by line and mode:", g.assign(line=g.subject.map(line_of)).groupby(["line", "mode"]).size().to_dict())
print("FS cells left out of the latency:", g[g.lat_centre.isna()].lat_excluded.value_counts().to_dict())

**Result: partially reproduced.** Over the 17 FS cells with a principal whisker (the claims file's
rule; the paper has 26), the nonwhisking rate (8.53 spikes/s) and the latency (5.23 ms with the bin
edge, 5.73 ms with the centre, over 13 cells) are within tolerance, but spikes per touch average
1.50 against 1.9 +- 0.2. The tag at p < 0.01 gives the same cells. The variant without the waveform
boundary (25 cells) gives 1.48 spikes per touch and a latency of 6.16 or 6.66 ms.

## fs-vm-touch-whisking

> In whole-cell L4 FS interneurons (n = 7), touch depolarizes the membrane by 7.0 +- 5.2 mV with onset 4.0 +- 0.6 ms, and whisking depolarizes it by 2.7 +- 2.1 mV toward threshold.

In [ ]:
for sel in ("all", "no_flagged"):
    for filt in yu2016.FILTERS:
        f_ = vm_table("FS", sel, filt=filt)
        print(f"[{sel}, {filt}] whole-cell FS cells: {len(f_)} (paper 7): touch amplitude mean {f_.amp_mv.mean():.2f} mV (expected 7.0 +- 1.97), "
              f"latency mean (expected 4.0 +- 0.4 ms) full rate {f_.latency_ms.mean():.2f}, 1 ms edge {f_.latency_1ms_edge.mean():.2f}, "
              f"centre {f_.latency_1ms_centre.mean():.2f}; whisking (a) onset change mean {f_.whisk_onset_change.mean():.2f} mV, "
              f"(b) threshold-distance shift mean {f_.shift_whisk.mean():.2f} mV (expected 2.7 +- 0.79)")
print(vm_table("FS", "all")[["subject", "amp_mv", "latency_ms", "latency_1ms_centre", "whisk_onset_change", "shift_whisk", "n_onsets", "touch_avg_n"]]
      .assign(line=lambda d: d.subject.map(line_of)).round(2).reset_index(drop=True).to_string())
print("Variant (not a reading of the claims file): the PV-ChR2 cell, FS in Supplementary Table 2, added:",
      vmm[(vmm.subject == "anm250509") & (vmm.filt == "filtfilt") & (vmm.O3 == "a")][["amp_mv", "latency_ms", "latency_1ms_centre", "whisk_onset_change", "shift_whisk"]].round(2).to_dict("records"))

**Result: not reproduced.** The dandiset holds 5 whole-cell FS cells by the claims file's rules
(the paper has 7). Their touch amplitude averages 5.56 mV (within 7.0 +- 1.97) and the whisking
depolarization 2.54 mV with the onset windows (within 2.7 +- 0.79; 1.01 mV with the threshold
distance), but the touch latency at the full sampling rate averages 1.88 ms against 4.0 +- 0.4, with
two cells at 0.0 and 0.3 ms. On the average in 1 ms bins, a resolution the claims file does not
name, the mean over 3 cells is 3.67 to 4.17 ms; this is asked as a question.

## fs-whisking-rate

> Most L4 FS interneurons (23 of 26) increase their spike rate during whisking without touch and 3 of 26 decrease it; over the population the rate roughly doubles (to 21.1 +- 15.8 spikes/s).

In [ ]:
def fs_rate_table(groups):
    rows = []
    for rname, col in groups.items():
        for sel in ("all", "no_flagged"):
            g = group("FS", None, sel, col)
            for f in yu2016.FILTERS:
                for o2 in ("O2a", "O2b"):
                    rows.append({"FS reading": rname, "cells": sel, "filter": f, "samples": o2, "n": len(g),
                                 "increase": int((g[f"{f}:{o2}"] == 1).sum()), "decrease": int((g[f"{f}:{o2}"] == -1).sum()),
                                 "nonwhisk_mean": g[f"{f}:rate_nonwhisk"].mean(), "whisk_mean": g[f"{f}:rate_whisk"].mean(),
                                 "whisk_sd": g[f"{f}:rate_whisk"].std(), "whisk_median": g[f"{f}:rate_whisk"].median(),
                                 "signed_rank_p": "%.2g" % wilcox(g[f"{f}:rate_nonwhisk"], g[f"{f}:rate_whisk"])})
    return pd.DataFrame(rows)
print("FS cells, trials without light; expected increases 23 +- 2.3 and decreases 3 +- 0.3 of 26, whisking mean 21.1 +- 3.1 spikes/s")
print(fs_rate_table({**FS_READINGS, **FS_VARIANT}).round(2).to_string(index=False))

**Result: partially reproduced.** The 17 FS cells fire more during whisking (signed-rank p =
0.00029 and 0.0005), with a mean whisking rate of 19.02 spikes/s (zero-phase, within 21.1 +- 3.1;
17.77 with the forward filter, outside). 13 cells increase and 1 or 2 decrease significantly against
the paper's 23 and 3 of 26, outside the tolerances; the group is smaller than the paper's.

## fs-whisking-phase

> Spikes of L4 FS interneurons are coupled to whisking phase.

In [ ]:
rows = []
for rname, col in {**FS_READINGS, **FS_VARIANT}.items():
    for sel in ("all", "no_flagged"):
        g = group("FS", None, sel, col)
        for f in yu2016.FILTERS:
            rows.append({"FS reading": rname, "cells": sel, "filter": f, "n": len(g), "coupled": int((g[f"{f}:r2"] > 0.25).sum()),
                         "double_fits": int((g[f"{f}:fit"] == "double").sum()), "r2_median": g[f"{f}:r2"].median(),
                         "strength_median": g[f"{f}:strength"].median(), "whisk_spikes_median": g[f"{f}:whisk_spikes"].median()})
print("FS cells coupled to whisking phase (r2 > 0.25; sinusoid, or with the second harmonic when its r2 is 1.5 times higher):")
print(pd.DataFrame(rows).round(2).to_string(index=False))
na = align[align.path.isin(group("FS").path)]
print(f"Reference (not part of the criterion): with spikes moved to the next trial (single sinusoid), {int((na.r2_shift > 0.25).sum())} "
      f"of {len(na)} FS cells have r2 > 0.25")

**Result: reproduced.** 16 of 17 FS cells are coupled to whisking phase (r2 > 0.25) under both
filters. As for VPM, 9 of the 17 also pass with spikes moved to the next trial, which the criterion
does not ask for.

## fs-touch-and-whisking

> Touch- and whisking-evoked spikes are typically multiplexed in individual L4 FS neurons, as in VPM neurons.

In [ ]:
rows = []
for rname, col in {**FS_READINGS, **FS_VARIANT, "VPM units": None}.items():
    g = group("VPM") if col is None else group("FS", None, "all", col)
    for f in yu2016.FILTERS:
        for o2 in ("O2a", "O2b"):
            mult = (g.spt_lo > 0) & (g[f"{f}:{o2}"] != 0)
            rows.append({"group": rname, "filter": f, "samples": o2, "n": len(g), "touch": int((g.spt_lo > 0).sum()),
                         "whisking": int((g[f"{f}:{o2}"] != 0).sum()), "both": int(mult.sum()), "fraction": mult.mean()})
print("Cells whose spikes per touch has a 95% interval above zero and whose whisking rate differs from the nonwhisking rate "
      "(rank-sum p < 0.05); more than half needed under every reading")
print(pd.DataFrame(rows).round(2).to_string(index=False))
for sel in ("no_flagged",):
    g = group("FS", None, sel)
    print(f"[{sel}] FS: both in", [int(((g.spt_lo > 0) & (g[f'{f}:{o}'] != 0)).sum()) for f in yu2016.FILTERS for o in ("O2a", "O2b")], "of", len(g))

**Result: reproduced.** 11 or 12 of the 17 FS cells (0.65 to 0.71) both respond to touch and change
their rate with whisking under every reading, more than half; the VPM units give 18 to 22 of 29
(0.62 to 0.76).

## exc-mainly-touch

> L4 excitatory neurons respond mostly to touch (0.3 +- 0.4 spikes per touch, n = 85) and only weakly to whisker movement; their rate does not change significantly between nonwhisking and whisking (0.4 to 0.6 spikes/s, P = 0.20).

In [ ]:
rows = []
for sel in ("all", "no_flagged"):
    g = group("E", None, sel)
    for f in yu2016.FILTERS:
        rows.append({"cells": sel, "filter": f, "n": len(g), "whole_cell": int((g["mode"] == "whole-cell").sum()), "spt_mean": g.spt.mean(),
                     "spt_sd": g.spt.std(), "spt_median": g.spt.median(), "nonwhisk_mean": g[f"{f}:rate_nonwhisk"].mean(),
                     "nonwhisk_median": g[f"{f}:rate_nonwhisk"].median(), "whisk_mean": g[f"{f}:rate_whisk"].mean(),
                     "whisk_median": g[f"{f}:rate_whisk"].median(), "whisk_minus_nonwhisk": (g[f"{f}:rate_whisk"] - g[f"{f}:rate_nonwhisk"]).mean(),
                     "signed_rank_p": "%.2g" % wilcox(g[f"{f}:rate_nonwhisk"], g[f"{f}:rate_whisk"]),
                     "whisk_higher": int((g[f"{f}:rate_whisk"] > g[f"{f}:rate_nonwhisk"]).sum())})
print("L4 E cells, trials without light; expected spikes per touch 0.3 +- 0.043, nonwhisking 0.4 +- 0.065, whisking 0.6 +- 0.098 spikes/s; "
      "absence judged by the signed-rank test at 0.05")
print(pd.DataFrame(rows).round(3).to_string(index=False))
g = group("E")
print("E cells by line and mode:", g.assign(line=g.subject.map(line_of)).groupby(["line", "mode"]).size().to_dict())
print("Rates by mode (filtfilt), means:", g.groupby("mode")[["spt", "filtfilt:rate_nonwhisk", "filtfilt:rate_whisk"]].mean().round(3).to_dict("index"))

**Result: not reproduced.** The 97 excitatory cells fire more during whisking than without it
(signed-rank p = 9.1e-07 and 7.4e-06), so the paper's absence of a change is not found. Spikes per
touch (0.319) is within tolerance of 0.3, but the nonwhisking (1.53 spikes/s) and whisking (3.55)
rates are well above 0.4 and 0.6; the 46 cell-attached cells fire more than the 51 whole-cell ones
(2.45 against 0.70 spikes/s without whisking).

## Light experiments

The functions below select the light trials and the windows used by the thalamic photoinhibition
and PV suppression claims (windows O5, fractions O6, laser powers O7).

In [ ]:
# Light designs. Thalamic photoinhibition: trials lit at the site labelled VPM and at no other site (VGAT-ChR2 mice).
# PV suppression: trials lit at the vS1 site of PV-IRES-Cre mice (constant light). Control trials have no light at
# any site. Windows O5: (a) control samples inside the median light span of the lit trials, (b) anywhere.
def light_design(r, region):
    tr = r["trials"]
    sites = [s for s, L in r["light"].items() if L["region"] == region and len(L["pulses"])]
    if not sites:
        return None
    s = sites[0]
    w = yu2016.light_windows(r["light"], s, tr)
    other = set()
    for s2 in r["light"]:
        if s2 != s:
            other |= set(yu2016.light_windows(r["light"], s2, tr).trial.astype(int))
    w = w[~w.trial.isin(other)]
    if len(w) == 0:
        return None
    P = r["light"][s]["pulses"]
    pk = yu2016.per_trial_index(P[:, 0], tr)
    pw = r["light"][s].get("pulse_power", np.ones(len(P)))
    power = {int(t): float(pw[pk == t].max()) for t in w.trial}
    return {"site": s, "windows": {int(t): (a, b) for t, a, b in zip(w.trial, w.rel_start, w.rel_stop)},
            "span": (float(w.rel_start.median()), float(w.rel_stop.median())), "power": power,
            "ctl": ~yu2016.lit_trials(r["light"], tr)}

def masks(r, d, reading, top_power=False):
    b, tr = r["beh"], r["trials"]
    tri = b["trial"]
    rel = b["t"] - tr.start.to_numpy()[np.clip(tri, 0, None)]
    lo, hi = np.full(len(tr), np.inf), np.full(len(tr), -np.inf)
    pmax = max(d["power"].values())
    for t, (a, c) in d["windows"].items():
        if not top_power or d["power"][t] == pmax:
            lo[t], hi[t] = a, c
    k = np.clip(tri, 0, None)
    use_l = (tri >= 0) & (rel >= lo[k]) & (rel < hi[k])
    use_c = (tri >= 0) & d["ctl"][k]
    if reading == "a":
        use_c &= (rel >= d["span"][0]) & (rel < d["span"][1])
    return use_l, use_c

def light_compare(r, spikes, d, filt="filtfilt", reading="a", definition="cortex5", min_iti=0.025, top_power=False):
    b = r["beh"]
    use_l, use_c = masks(r, d, reading, top_power)
    count = yu2016.counts_1k(spikes, b)
    out = {}
    on = yu2016.touches(b, None, min_iti)
    for tag, u in (("light", use_l), ("ctl", use_c)):
        er = yu2016.epoch_rates(np.where(u, count, 0), b, filt, u)
        out[f"{tag}_nonwhisk"], out[f"{tag}_whisk"] = er["rate_nonwhisk"], er["rate_whisk"]
        out[f"{tag}_t_nonwhisk"], out[f"{tag}_t_whisk"] = er["time_nonwhisk"], er["time_whisk"]
        s = yu2016.spikes_per_touch(count, b["trial"], on[u[on]], definition)
        out[f"{tag}_spt"], out[f"{tag}_touches"] = s["spt"], s["n_touches"]
        out[f"{tag}_rate"] = count[u].sum() / max(u.sum(), 1) * 1000
    return out

def frac(light, ctl, how):
    light, ctl = np.asarray(light, float), np.asarray(ctl, float)
    ok = np.isfinite(light) & np.isfinite(ctl)
    if how == "a":
        return light[ok].mean() / ctl[ok].mean()
    ok &= ctl > 0
    return float(np.median(light[ok] / ctl[ok]))

## thalamus-drives-fs

> Photoinhibiting the thalamus reduces the spike rates of L4 FS interneurons during nonwhisking (to 4% of control), whisking (to 13%) and touch (to 32%), so FS interneurons are driven by thalamic input.

In [ ]:
def thal_rows(col):
    out = []
    for r in l4:
        c = cells.set_index("path").loc[r["path"]]
        if c[col] != "FS" or c.line != "VGAT-ChR2-EYFP" or r["beh"]["principal"] is None:
            continue
        d = light_design(r, "VPM")
        if d is None:
            continue
        for filt in yu2016.FILTERS:
            for reading in ("a", "b"):
                for top in (False, True):
                    out.append({"path": r["path"], "subject": r["subject"], "filter": filt, "O5": reading, "O7": "b" if top else "a",
                                "lit_trials": len(d["windows"]), "powers": sorted(set(d["power"].values())),
                                **light_compare(r, r["spikes"], d, filt, reading, top_power=top)})
    return pd.DataFrame(out)

def thal_summary(t):
    rows = []
    for (filt, o5, o7), g in t.groupby(["filter", "O5", "O7"]):
        row = {"filter": filt, "O5": o5, "O7": o7, "cells": len(g)}
        for m in ("nonwhisk", "whisk", "spt"):
            gg = g.dropna(subset=[f"light_{m}", f"ctl_{m}"])
            row.update({f"{m}_n": len(gg), f"{m}_O6a": frac(gg[f"light_{m}"], gg[f"ctl_{m}"], "a"),
                        f"{m}_O6b": frac(gg[f"light_{m}"], gg[f"ctl_{m}"], "b"),
                        f"{m}_lower": int((gg[f"light_{m}"] < gg[f"ctl_{m}"]).sum()), f"{m}_p": "%.2g" % wilcox(gg[f"light_{m}"], gg[f"ctl_{m}"])})
        rows.append(row)
    return pd.DataFrame(rows)

th = thal_rows("cls")
print("FS cells with thalamic light:", th.path.nunique(), "from mice", sorted(th.subject.unique()), "; lit trials per cell",
      th.drop_duplicates("path").lit_trials.tolist(), "; laser powers (stored values)", th.drop_duplicates("path").powers.tolist())
print("Expected fractions of control: nonwhisking 0.04 +- 0.004, whisking 0.13 +- 0.013, spikes per touch 0.32 +- 0.032; each test p < 0.05 with light lower")
print(thal_summary(th).round(3).to_string(index=False))
tv = thal_rows("cls_tag")
print(f"\nVariant (tag only, any waveform): {tv.path.nunique()} cells")
print(thal_summary(tv)[lambda d: d.O7 == "a"].round(3).to_string(index=False))
print("Per cell (filtfilt, O5 a, O7 a):")
print(th[(th["filter"] == "filtfilt") & (th.O5 == "a") & (th.O7 == "a")][["subject", "ctl_nonwhisk", "light_nonwhisk", "ctl_whisk", "light_whisk",
      "ctl_spt", "light_spt", "light_touches", "light_t_nonwhisk", "light_t_whisk"]].round(3).to_string(index=False))

**Result: blocked (too few cells).** Only 5 FS cells by the claims file's rule were recorded with
thalamic light, all from one mouse, and with 5 cells a two-sided signed-rank test cannot reach p <
0.05 (smallest p 0.0625). The rates fall with the light in all 5 cells without whisking and in 4 of
5 during whisking, to 0.20 and 0.27 of control (population means, all powers), and spikes per touch
to 0.41; at the highest power only the nonwhisking rate falls to 0.022 of control. The fractions are
above the paper's 0.04, 0.13 and 0.32 except at the highest power without whisking. The variant
without the waveform boundary (9 cells) passes all three tests (p = 0.012, 0.0078 and 0.0039).

## exc-thalamus-photoinhibition

> Thalamic photoinhibition hyperpolarizes L4 excitatory neurons and reduces their Vm fluctuations, with an onset latency of 4.2 ms, longer than the 2.8 ms latency of direct cortical GABAergic activation.

In [ ]:
from scipy import signal as sg
bh, ah = sg.butter(2, 10 / 500, "high")
def vm_light(r, d, reading):
    b = r["beh"]
    use_l, use_c = masks(r, d, reading)
    vm = r["vm1k"].astype(float)
    hp = np.full(len(vm), np.nan)
    for k in np.unique(b["trial"][b["trial"] >= 0]):
        i = np.flatnonzero(b["trial"] == k)
        x = vm[i]
        g = np.isfinite(x)
        if g.sum() > 100:
            hp[i] = sg.filtfilt(bh, ah, np.interp(np.arange(len(x)), np.flatnonzero(g), x[g]))
            hp[i[~g]] = np.nan
    keep = ~b["excl"] & np.isfinite(vm)
    return {f"{t}_{q}": f(x[u & keep]) for t, u in (("light", use_l), ("ctl", use_c)) for q, x, f in (("vm", vm, np.mean), ("var", hp, np.nanvar))}

def resample(avg, fs):
    return avg if fs == 10000 else avg[::int(fs // 10000)]

rows, avgs = [], []
for r in l4:
    c = cells.set_index("path").loc[r["path"]]
    if c.cls != "E" or r["mode"] != "whole-cell" or c.line != "VGAT-ChR2-EYFP" or "vm1k" not in r:
        continue
    d = light_design(r, "VPM")
    if d is None:
        continue
    for reading in ("a", "b"):
        rows.append({"path": r["path"], "subject": r["subject"], "O5": reading, "lit_trials": len(d["windows"]), **vm_light(r, d, reading)})
    la = r["light_avg"].get(d["site"], {}).get("first", (None, 0))
    if la[0] is not None:
        avgs.append(resample(la[0], r["vm_hz"]) - resample(la[0], r["vm_hz"])[:int(0.05 * 10000)].mean())
tp = pd.DataFrame(rows)
print("Whole-cell E cells with thalamic light:", tp.path.nunique(), "(paper 12), from", tp.subject.nunique(), "mice; lit trials per cell",
      tp.drop_duplicates("path").lit_trials.tolist())
for reading, g in tp.groupby("O5"):
    print(f"O5 {reading}: Vm lower with light in {(g.light_vm < g.ctl_vm).sum()} of {len(g)} (median change {np.median(g.light_vm - g.ctl_vm):.2f} mV, "
          f"signed-rank p = {wilcox(g.light_vm, g.ctl_vm):.2g}); variance (>10 Hz) lower in {(g.light_var < g.ctl_var).sum()} "
          f"(median ratio {np.median(g.light_var / g.ctl_var):.2f}, p = {wilcox(g.light_var, g.ctl_var):.2g})")
ga = np.mean(avgs, axis=0) if avgs else None
lat_thal = yu2016.neg_onset_latency(ga, len(avgs), 10000)
lat_thal_1ms = yu2016.neg_onset_latency(ga, len(avgs), 10000, binned=True)
print(f"Grand average aligned to thalamic laser onset ({len(avgs)} cells): onset latency of the hyperpolarization {lat_thal:.2f} ms at full rate, {lat_thal_1ms:.1f} ms in 1 ms bins (expected 4.2 +- 0.42); "
      f"mean change 10 to 50 ms after onset {ga[int(0.06 * 1e4):int(0.1 * 1e4)].mean():.2f} mV")
cav = []
for r in l4:
    c = cells.set_index("path").loc[r["path"]]
    if c.cls != "E" or r["mode"] != "whole-cell" or c.line != "VGAT-ChR2-EYFP" or "vm1k" not in r:
        continue
    for s, L in r["light"].items():
        if "vS1" in L["region"]:
            a, n = r["light_avg"].get(s, {}).get("isolated", (None, 0))
            if a is not None and n >= 5:
                x = resample(a, r["vm_hz"])
                cav.append(x - x[:int(0.05 * 1e4)].mean())
gc = np.mean(cav, axis=0) if cav else None
lat_ctx = yu2016.neg_onset_latency(gc, len(cav), 10000)
lat_ctx_1ms = yu2016.neg_onset_latency(gc, len(cav), 10000, binned=True)
print(f"Grand average aligned to isolated cortical pulses ({len(cav)} whole-cell E cells in VGAT-ChR2 mice; paper 7): onset latency {lat_ctx:.2f} ms at full rate, {lat_ctx_1ms:.1f} ms in 1 ms bins "
      f"(expected 2.8 +- 0.28); mean change 5 to 20 ms after onset {gc[int(0.055 * 1e4):int(0.07 * 1e4)].mean():.2f} mV")
print("Grand averages 0 to 8 ms after onset, 0.5 ms steps (mV): thalamic", np.round(ga[500:581:5], 2).tolist())
print("   cortical", np.round(gc[500:581:5], 2).tolist())
fig, ax = plt.subplots(1, 2, figsize=(9, 2.8), sharey=True)
tt = (np.arange(len(ga)) / 1e4 - 0.05) * 1000
ax[0].plot(tt, ga, color="k", lw=0.8); ax[0].axvline(0, color="C0"); ax[0].set_title("thalamic light", fontsize=8)
ax[1].plot(tt, gc, color="k", lw=0.8); ax[1].axvline(0, color="C0"); ax[1].set_title("isolated cortical pulse", fontsize=8)
for a in ax:
    a.set_xlim(-20, 60); a.set_xlabel("ms from laser onset")
ax[0].set_ylabel("Vm change (mV)"); plt.tight_layout(); plt.show()
print("Plotted range -20 to 60 ms; values (mV):", round(float(ga[300:1100].min()), 2), "to", round(float(ga[300:1100].max()), 2), "and",
      round(float(gc[300:1100].min()), 2), "to", round(float(gc[300:1100].max()), 2))

**Result: partially reproduced.** Thalamic light hyperpolarizes 12 of 13 whole-cell excitatory
cells (median -4.02 mV) and lowers their Vm variance above 10 Hz in 10 (median ratio 0.55), under
both window readings. The onset latency at the full sampling rate is 0.10 ms for both the thalamic
and the direct cortical light, against 4.2 and 2.8 ms, so the thalamic latency is not longer; in 1
ms bins, a resolution the claims file does not name, it is 5.0 against 2.0 ms.

## pv-enphr-suppression

> Yellow light in PV-IRES-Cre mice expressing eNpHR3.0 strongly inhibits most L4 PV interneurons (3 of 4); their rate falls 3 ms after laser onset and L4 excitatory activity rises 2 ms later.

In [ ]:
def boot_rate_ci(count, b, filt, use, rng):
    """95% bootstrap interval (1,000 draws over trials) of the rate in nonwhisking and whisking samples."""
    out = {}
    tri = b["trial"]
    for name in ("nonwhisk", "whisk"):
        m = b[filt][name] & use & (tri >= 0)
        ks = tri[m]
        n = tri.max() + 1
        c = np.bincount(ks, weights=count[m], minlength=n)
        t = np.bincount(ks, minlength=n) / 1000
        keep = t > 0
        c, t = c[keep], t[keep]
        if len(c) < 2:
            out[name] = (np.nan, np.nan)
            continue
        idx = rng.integers(0, len(c), (yu2016.N_BOOT, len(c)))
        bs = c[idx].sum(axis=1) / t[idx].sum(axis=1)
        out[name] = (np.percentile(bs, 2.5), np.percentile(bs, 97.5))
    return out

def onset_psth(r, d):
    """Rate (spikes/s) in 1 ms bins from 25 ms before to 100 ms after the first light sample of each lit trial."""
    sp = np.sort(r["spikes"])
    st = r["trials"].start.to_numpy()
    on = np.array([st[t] + w[0] for t, w in d["windows"].items()])
    edges = np.arange(-25, 101) / 1000
    h = np.zeros(len(edges) - 1)
    for o in on:
        h += np.histogram(sp[(sp >= o - 0.025) & (sp < o + 0.1)] - o, edges)[0]
    return h / len(on) * 1000

def psth_crossing(m, sign):
    lags = np.arange(-25, 100)
    dd = np.diff(m)
    lag_d = lags[1:]
    thr = 3 * dd[(lag_d >= -25) & (lag_d < 0)].std()
    hit = np.flatnonzero((lag_d >= 0) & (sign * dd > thr))
    return (float(lag_d[hit[0]]), float(lag_d[hit[0]] + 0.5)) if len(hit) else (np.nan, np.nan)

rng = np.random.default_rng(13)
rows, ps_pv, ps_e = [], [], []
for r in l4:
    c = cells.set_index("path").loc[r["path"]]
    if c.line != "PV-IRES-Cre" or r["beh"]["principal"] is None:
        continue
    d = light_design(r, "vS1")
    if d is None:
        continue
    ps = onset_psth(r, d)
    if c.cls == "FS":
        ps_pv.append(ps)
        b = r["beh"]
        use_l, use_c = masks(r, d, "a")
        count = yu2016.counts_1k(r["spikes"], b)
        ci = boot_rate_ci(count, b, "filtfilt", use_c, rng)
        lr = light_compare(r, r["spikes"], d, "filtfilt", "a")
        rows.append({"path": r["path"].split("/")[1][:40], "ctl_nonwhisk": lr["ctl_nonwhisk"], "ctl_nonwhisk_lo": ci["nonwhisk"][0],
                     "light_nonwhisk": lr["light_nonwhisk"], "ctl_whisk": lr["ctl_whisk"], "ctl_whisk_lo": ci["whisk"][0],
                     "light_whisk": lr["light_whisk"],
                     "strongly_inhibited": lr["light_nonwhisk"] < ci["nonwhisk"][0] and lr["light_whisk"] < ci["whisk"][0]})
    elif c.cls == "E":
        ps_e.append(ps)
pvr = pd.DataFrame(rows)
print("PV cells identified (PV-IRES-Cre, rate lower with the light):", len(pvr), "(paper 4); strongly inhibited (light rate below the lower "
      "bound of the control 95% interval in nonwhisking and whisking periods):", int(pvr.strongly_inhibited.sum()))
print(pvr.round(2).to_string(index=False))
gpv, ge = np.mean(ps_pv, axis=0), np.mean(ps_e, axis=0)
lpv, le_ = psth_crossing(gpv, -1), psth_crossing(ge, +1)
print(f"Grand average PSTH aligned to laser onset: PV cells ({len(ps_pv)}) rate falls at {lpv[0]:.1f} ms (bin edge) / {lpv[1]:.1f} ms (centre), "
      f"expected 3 +- 0.3; E cells ({len(ps_e)}) rate rises at {le_[0]:.1f} / {le_[1]:.1f} ms, a further delay of {le_[0] - lpv[0]:.1f} ms "
      f"(expected 2 +- 0.2)")
print("PV grand average, 0 to 10 ms (spikes/s):", np.round(gpv[25:36], 1).tolist(), "; 25 ms before: mean", round(gpv[:25].mean(), 1))
print("E grand average, 0 to 10 ms (spikes/s):", np.round(ge[25:36], 2).tolist(), "; 25 ms before: mean", round(ge[:25].mean(), 2))

**Result: partially reproduced.** 3 PV cells are identified by their inactivation, and all 3 are
strongly inhibited (their light rate below the control interval in both periods); the paper's fourth
PV cell cannot be found, and since the cells are found by their inactivation the count is partly
circular. The dPSTH/dt rule finds no fall in the grand average of the 3 PV cells, and the excitatory
cells' rise is found at 48 ms, so neither latency is within tolerance.

## fs-suppress-movement-spiking

> Partially suppressing PV (FS) interneurons increases the spike rate of L4 excitatory neurons during nonwhisking (0.6 to 1.6 spikes/s) and whisking (0.8 to 2.7 spikes/s).

In [ ]:
def pv_rows(min_iti=0.025, definition="cortex5"):
    out = []
    for r in l4:
        c = cells.set_index("path").loc[r["path"]]
        if c.line != "PV-IRES-Cre" or c.cls != "E" or r["beh"]["principal"] is None:
            continue
        d = light_design(r, "vS1")
        if d is None:
            continue
        for filt in yu2016.FILTERS:
            for reading in ("a", "b"):
                out.append({"path": r["path"], "subject": r["subject"], "mode": r["mode"], "filter": filt, "O5": reading, "lit_trials": len(d["windows"]),
                            **light_compare(r, r["spikes"], d, filt, reading, definition, min_iti)})
    return pd.DataFrame(out)

pvE = pv_rows()
pvE8 = pv_rows(0.010, "cortex8")
print("E cells in PV-IRES-Cre mice with the eNpHR light:", pvE.path.nunique(), "(paper 40 of 43), from", pvE.subject.nunique(), "mice;",
      pvE.drop_duplicates("path")["mode"].value_counts().to_dict())
rows = []
for (filt, o5), g in pvE.groupby(["filter", "O5"]):
    g = g[(g.light_t_nonwhisk > 0) & (g.light_t_whisk > 0) & (g.ctl_t_nonwhisk > 0) & (g.ctl_t_whisk > 0)]
    kw = stats.kruskal(g.ctl_nonwhisk, g.ctl_whisk, g.light_nonwhisk, g.light_whisk).pvalue
    rows.append({"filter": filt, "O5": o5, "n": len(g), "ctl_nonwhisk": g.ctl_nonwhisk.mean(), "light_nonwhisk": g.light_nonwhisk.mean(),
                 "ctl_whisk": g.ctl_whisk.mean(), "light_whisk": g.light_whisk.mean(),
                 "sem_light_nonwhisk": g.light_nonwhisk.sem(), "sem_light_whisk": g.light_whisk.sem(),
                 "nonwhisk_higher": int((g.light_nonwhisk > g.ctl_nonwhisk).sum()), "p_nonwhisk": "%.2g" % wilcox(g.light_nonwhisk, g.ctl_nonwhisk),
                 "whisk_higher": int((g.light_whisk > g.ctl_whisk).sum()), "p_whisk": "%.2g" % wilcox(g.light_whisk, g.ctl_whisk),
                 "kruskal_wallis_p": "%.2g" % kw})
print("Expected means: nonwhisking 0.6 +- 0.1 -> 1.6 +- 0.3, whisking 0.8 +- 0.2 -> 2.7 +- 0.5 spikes/s; both paired tests p < 0.05 with light higher")
print(pd.DataFrame(rows).round(3).to_string(index=False))
g = pvE[(pvE["filter"] == "filtfilt") & (pvE.O5 == "a")]
print("Cells dropped for lack of an epoch of each kind (filtfilt, O5 a):",
      int(((g.light_t_nonwhisk == 0) | (g.light_t_whisk == 0) | (g.ctl_t_nonwhisk == 0) | (g.ctl_t_whisk == 0)).sum()))
print("By mouse (filtfilt, O5 a), mean rates:", g.groupby("subject")[["ctl_nonwhisk", "light_nonwhisk", "ctl_whisk", "light_whisk"]].mean().round(2).to_dict("index"))
print("Seconds per cell inside the light, median: nonwhisking", round(g.light_t_nonwhisk.median(), 1), ", whisking", round(g.light_t_whisk.median(), 1))

**Result: partially reproduced.** In the 42 excitatory cells of PV-IRES-Cre mice the light raises
the nonwhisking rate (33 cells, p = 6.2e-05) and the whisking rate (33 cells, p = 7.4e-05) under every
reading, but the rates are higher than the paper's: 1.26 to 2.26 spikes/s without whisking (paper 0.6
to 1.6) and 3.95 to 6.14 with it (paper 0.8 to 2.7), outside the tolerances.

## fs-enhance-touch-selectivity

> With PV interneurons suppressed, L4 excitatory neurons become driven by whisker movement (whisking rate above nonwhisking, P = 0.02), while in control trials they are not (P = 0.55); their selectivity for touch is normally maintained in part by FS interneurons.

In [ ]:
rows = []
for (filt, o5), g in pvE.groupby(["filter", "O5"]):
    g = g[(g.light_t_nonwhisk > 0) & (g.light_t_whisk > 0) & (g.ctl_t_nonwhisk > 0) & (g.ctl_t_whisk > 0)]
    rows.append({"filter": filt, "O5": o5, "n": len(g), "light_whisk_minus_nonwhisk": (g.light_whisk - g.light_nonwhisk).mean(),
                 "light_whisk_higher": int((g.light_whisk > g.light_nonwhisk).sum()), "p_light": "%.2g" % wilcox(g.light_whisk, g.light_nonwhisk),
                 "ctl_whisk_minus_nonwhisk": (g.ctl_whisk - g.ctl_nonwhisk).mean(), "ctl_whisk_higher": int((g.ctl_whisk > g.ctl_nonwhisk).sum()),
                 "p_ctl": "%.2g" % wilcox(g.ctl_whisk, g.ctl_nonwhisk)})
print("Whisking against nonwhisking rate within light trials (paper P = 0.02) and within control trials (paper P = 0.55):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: partially reproduced.** With the light, whisking rates exceed nonwhisking rates in 33 of
42 cells (p = 2.2e-06), as the paper reports, but in control trials they do too (31 of 42, p =
3.3e-05), where the paper found no difference (P = 0.55).

## exc-touch-spikes-enphr

> PV interneuron suppression does not change mean touch-evoked spiking of L4 excitatory neurons significantly (0.27 to 0.32 spikes per touch, P = 0.09); its effect is heterogeneous, with weakly responding neurons responding more and strongly responding neurons unchanged or less.

In [ ]:
rows = []
for (filt, o5), g in pvE8[pvE8["filter"] == "filtfilt"].groupby(["filter", "O5"]):
    g = g.dropna(subset=["ctl_spt", "light_spt"])
    rows.append({"O5": o5, "n": len(g), "ctl_mean": g.ctl_spt.mean(), "ctl_sem": g.ctl_spt.sem(), "light_mean": g.light_spt.mean(),
                 "light_sem": g.light_spt.sem(), "higher": int((g.light_spt > g.ctl_spt).sum()), "p": "%.2g" % wilcox(g.light_spt, g.ctl_spt),
                 "touches_light_median": g.light_touches.median()})
print("Spikes per touch (Fig. 8e definition, ITI >= 10 ms), light against control; expected 0.27 +- 0.06 -> 0.32 +- 0.06, P = 0.09 (absence at 0.05)")
print(pd.DataFrame(rows).round(3).to_string(index=False))
g = pvE8[(pvE8["filter"] == "filtfilt") & (pvE8.O5 == "a")].dropna(subset=["ctl_spt", "light_spt"])
lo = g.ctl_spt < g.ctl_spt.median()
print(f"Heterogeneity (described, not tested): cells below the median control response change by {np.mean(g.light_spt[lo] - g.ctl_spt[lo]):+.3f} on average "
      f"({int((g.light_spt[lo] > g.ctl_spt[lo]).sum())} of {int(lo.sum())} rise); cells above it by {np.mean(g.light_spt[~lo] - g.ctl_spt[~lo]):+.3f} "
      f"({int((g.light_spt[~lo] > g.ctl_spt[~lo]).sum())} of {int((~lo).sum())} rise)")

**Result: not reproduced.** Spikes per touch (Fig. 8e definition) rise with the light from 0.339
to 0.448 (40 cells, p = 0.044) or from 0.304 to 0.442 (41 cells, p = 0.0091), so the paper's absence of
a significant change is not found. Cells below the median control response rise by 0.123 on average
and cells above it by 0.095.

## auc-weak-strong

> Suppressing PV interneurons improves touch detection (AUC) for L4 excitatory neurons with weak touch responses and diminishes it for neurons with robust touch responses.

In [ ]:
def auc_inputs(r, d, rng, reading):
    """Per trial, spike counts in the 250 ms before and after the first principal-whisker touch, or (no touch) in two
    adjacent 250 ms windows of the sample epoch: reading (a) at a random time, (b) around the median first-touch time
    of the touch trials. Light trials count when the 500 ms lie inside the light."""
    b, tr = r["beh"], r["trials"]
    sp = np.sort(r["spikes"])
    st = tr.start.to_numpy()
    on_t = b["t"][b["touch_on"]]
    k = yu2016.per_trial_index(on_t, tr)
    first = {}
    for t, kk in zip(on_t, k):
        if kk >= 0 and kk not in first:
            first[kk] = t
    med = np.median([first[kk] - st[kk] for kk in first]) if first else np.nan
    out = {"ctl": {"t": [[], []], "n": [[], []]}, "light": {"t": [[], []], "n": [[], []]}}
    cnt = lambda a, z: int(np.searchsorted(sp, z) - np.searchsorted(sp, a))
    for kk in range(len(tr)):
        if d["ctl"][kk]:
            cond, win = "ctl", None
        elif kk in d["windows"]:
            cond, win = "light", d["windows"][kk]
        else:
            continue
        if kk in first:
            t0, key = first[kk], "t"
        else:
            p0, p1 = tr.pole_in_time.iloc[kk], tr.pole_out_time.iloc[kk]
            if not (np.isfinite(p0) and np.isfinite(p1)) or p1 - p0 < 0.5:
                continue
            t0 = rng.uniform(p0 + 0.25, p1 - 0.25) if reading == "a" else st[kk] + med
            key = "n"
        if win is not None and not (st[kk] + win[0] <= t0 - 0.25 and t0 + 0.25 <= st[kk] + win[1]):
            continue
        if t0 - 0.25 < st[kk] or t0 + 0.25 > st[kk] + yu2016.ANALYSIS_END_S:
            continue
        out[cond][key][0].append(cnt(t0 - 0.25, t0))
        out[cond][key][1].append(cnt(t0, t0 + 0.25))
    return out

auc_rows = []
for reading in ("a", "b"):
    rng = np.random.default_rng(29 if reading == "a" else 31)
    for r in l4:
        c = cells.set_index("path").loc[r["path"]]
        if c.line != "PV-IRES-Cre" or c.cls != "E" or r["beh"]["principal"] is None:
            continue
        d = light_design(r, "vS1")
        if d is None:
            continue
        x = auc_inputs(r, d, rng, reading)
        row = {"path": r["path"], "subject": r["subject"], "reading": reading}
        for cond in ("ctl", "light"):
            tb, ta = np.array(x[cond]["t"][0]), np.array(x[cond]["t"][1])
            nb, na = np.array(x[cond]["n"][0]), np.array(x[cond]["n"][1])
            row[f"{cond}_touch_trials"], row[f"{cond}_notouch_trials"] = len(tb), len(nb)
            row[f"{cond}_auc"] = yu2016.roc_auc((tb, nb), (ta, na), rng)
            row[f"{cond}_after"] = ta.mean() / 0.25 if len(ta) else np.nan
            row[f"{cond}_before"] = tb.mean() / 0.25 if len(tb) else np.nan
        auc_rows.append(row)
auc = pd.DataFrame(auc_rows)
auc["evoked_ctl"] = auc.ctl_after - auc.ctl_before
auc["auc_change"] = auc.light_auc - auc.ctl_auc
for reading, g in auc.groupby("reading"):
    g = g.dropna(subset=["ctl_auc", "light_auc"])
    up, dn = g[g.auc_change > 0], g[g.auc_change < 0]
    print(f"Reading {reading}: {len(g)} cells with an AUC in both conditions (trials per cell, median: control {int(g.ctl_touch_trials.median())} touch, "
          f"{int(g.ctl_notouch_trials.median())} no-touch; light {int(g.light_touch_trials.median())} touch, {int(g.light_notouch_trials.median())} no-touch). "
          f"AUC rises in {len(up)}, falls in {len(dn)}, unchanged in {int((g.auc_change == 0).sum())}. Control touch-evoked rate (0 to 250 ms after minus "
          f"before, spikes/s): rising group mean {up.evoked_ctl.mean():.2f}, falling group mean {dn.evoked_ctl.mean():.2f} "
          f"(rank-sum p = {stats.ranksums(up.evoked_ctl, dn.evoked_ctl).pvalue:.2g}, reported only)")
    print(f"   AUC control median {g.ctl_auc.median():.3f}, light median {g.light_auc.median():.3f}")
    print(f"   Spearman correlation of AUC change with the control touch-evoked rate: rho = {stats.spearmanr(g.evoked_ctl, g.auc_change)[0]:.2f} (reported only)")
print("Cells without an AUC in a condition:", auc[auc.ctl_auc.isna() | auc.light_auc.isna()].groupby("reading").size().to_dict())

**Result: reproduced.** Cells whose AUC rises with the light have a lower control touch-evoked
rate than those whose AUC falls under both readings: 5.93 against 10.87 spikes/s (a) and 8.14
against 8.18 (b). The difference under reading (b) is very small, and no test separates the groups
under either reading (rank-sum p = 0.37 and 1, reported only).

## auc-groups

> Neurons whose AUC rose with PV suppression gained touch-evoked spiking (3.4 to 8.7 spikes/s); neurons whose AUC fell kept their touch-evoked response (7.3 to 7.0 spikes/s) but had higher baseline rates (0.9 to 2.6 spikes/s).

In [ ]:
rows = []
for reading, g in auc.groupby("reading"):
    g = g.dropna(subset=["ctl_auc", "light_auc"])
    up, dn = g[g.auc_change > 0], g[g.auc_change < 0]
    rows.append({"reading": reading, "n_up": len(up), "n_down": len(dn),
                 "up_after_ctl": up.ctl_after.mean(), "up_after_light": up.light_after.mean(), "p_up_after": "%.2g" % wilcox(up.ctl_after, up.light_after),
                 "down_after_ctl": dn.ctl_after.mean(), "down_after_light": dn.light_after.mean(), "p_down_after": "%.2g" % wilcox(dn.ctl_after, dn.light_after),
                 "down_before_ctl": dn.ctl_before.mean(), "down_before_light": dn.light_before.mean(), "p_down_before": "%.2g" % wilcox(dn.ctl_before, dn.light_before)})
print("Expected: increased-AUC group (19 +- 1.9) 3.4 +- 0.8 -> 8.7 +- 1.9 spikes/s, P = 0.0001; reduced-AUC group (21 +- 2.1) 7.3 +- 1.5 -> 7.0 +- 1.4 "
      "(no change, P = 0.98); its baseline 0.9 +- 0.2 -> 2.6 +- 0.6 (P = 0.0002)")
print(pd.DataFrame(rows).round(3).T.to_string())

**Result: blocked (methods underspecified).** The two readings of the no-touch windows give
different outcomes. Under (a) the three tests hold (rise in the increased-AUC group, p = 4.9e-05; no
change in the reduced-AUC group's touch response, p = 0.11; rise of its baseline, p = 0.0049), but
the group sizes (23 and 12) and the rates are outside tolerance, so it is partially reproduced. Under
(b) the reduced-AUC group's touch response changes (p = 0.021), so it is not reproduced.

## behavior-false-alarms

> With PV interneurons suppressed, most mice (6 of 8) are more likely to report touch with the pole out of reach (false alarms on no-lick trials).

In [ ]:
rows = []
for r in l4:
    if line_of[r["subject"]] != "PV-IRES-Cre":
        continue
    d = light_design(r, "vS1")
    if d is None:
        continue
    tr = r["trials"]
    lit = np.zeros(len(tr), bool)
    lit[list(d["windows"])] = True
    nogo = tr.type.str.lower().to_numpy() == "nogo"
    cr = tr.response.to_numpy() == "CR"
    for cond, m in (("light", lit), ("ctl", d["ctl"])):
        rows.append({"subject": r["subject"], "path": r["path"], "cond": cond, "nogo": int((m & nogo).sum()), "correct": int((m & nogo & cr).sum())})
bt = pd.DataFrame(rows)

def chi(g):
    t = g.groupby("cond")[["nogo", "correct"]].sum()
    if not {"light", "ctl"} <= set(t.index):
        return pd.Series({"ctl_correct": np.nan})
    tab = np.array([[t.loc[c, "correct"], t.loc[c, "nogo"] - t.loc[c, "correct"]] for c in ("ctl", "light")])
    ok = tab.sum(axis=1).min() > 0 and tab.sum(axis=0).min() > 0
    p_y = stats.chi2_contingency(tab)[1] if ok else np.nan
    p_n = stats.chi2_contingency(tab, correction=False)[1] if ok else np.nan
    return pd.Series({"ctl_nogo": int(tab[0].sum()), "ctl_correct": tab[0, 0] / tab[0].sum(), "light_nogo": int(tab[1].sum()),
                      "light_correct": tab[1, 0] / tab[1].sum() if tab[1].sum() else np.nan, "p_yates": p_y, "p_no_correction": p_n})

pooled = bt.groupby("subject").apply(chi)
pooled["decrease_yates"] = (pooled.p_yates < 0.05) & (pooled.light_correct < pooled.ctl_correct)
pooled["decrease_plain"] = (pooled.p_no_correction < 0.05) & (pooled.light_correct < pooled.ctl_correct)
pooled["increase_yates"] = (pooled.p_yates < 0.05) & (pooled.light_correct > pooled.ctl_correct)
print("PV-IRES-Cre mice with the eNpHR light:", len(pooled), "(paper 8 tested). Reading (a), each mouse's trials pooled; correct = CR on no-lick trials:")
print(pooled.round(3).to_string())
print("Mice with a significant decrease (Yates-corrected chi-squared):", int(pooled.decrease_yates.sum()), "; without the correction:",
      int(pooled.decrease_plain.sum()), "; significant increase:", int(pooled.increase_yates.sum()))
per = bt.groupby(["subject", "path"]).apply(chi).reset_index()
per["decrease"] = (per.p_yates < 0.05) & (per.light_correct < per.ctl_correct)
per["decrease_plain"] = (per.p_no_correction < 0.05) & (per.light_correct < per.ctl_correct)
ps = per.groupby("subject").agg(sessions=("path", "size"), any_decrease=("decrease", "any"), any_decrease_plain=("decrease_plain", "any"))
print("Reading (b), per session (file), a mouse counting when any session shows a significant decrease: Yates", int(ps.any_decrease.sum()),
      "; without the correction", int(ps.any_decrease_plain.sum()), "of", len(ps))

**Result: blocked (methods underspecified).** The claims file does not say whether the chi-squared
test is corrected for continuity, and the readings give different outcomes: with the Yates
correction, 4 of 9 mice (pooled) or 5 (any session) show a significant decrease of no-lick
performance; without it, 5 or 6. Under the criterion this is not reproduced, partially reproduced or
reproduced depending on the reading. The dandiset holds 9 PV-IRES-Cre mice with the light, the paper
tested 8. The question is asked.

## Blocked claims

> `exc-inhibition-timing`: With L4 excitatory neurons depolarized near 0 mV to reveal inhibition,
> touch evokes strong, transient inhibition that follows excitation by 0.4 +- 0.8 ms at half maximum.

> `exc-inhibition-whisking`: In L4 excitatory neurons held near 0 mV, inhibitory inputs occur
> spontaneously but are more prevalent at the onset of whisker movement; after whisking onset,
> excitation and inhibition increase gradually, with indistinguishable time courses, over about 100 ms.

> `vpm-photoinhibition`: Photoactivating TRN axons in VPM of VGAT-ChR2-EYFP mice reduces thalamic
> activity to 17% of control.

> `rabies-vpm-input`: Monosynaptic rabies tracing from L4 PV interneurons labels thalamic neurons
> mostly in VPM (>90%) and few in POM (<10%).

> `ion-cut-exc-no-touch`, `ion-cut-exc-whisking`, `ion-cut-exc-phase`, `ion-cut-fs-no-touch`,
> `sensory-input-drives-fs`, `ion-cut-fs-phase`, `ion-cut-behavior`: after infraorbital nerve
> transection, L4 excitatory neurons do not depolarize after touch, their whisking depolarization and
> phase locking are reduced, touch does not evoke spikes in FS cells, their whisking activation is
> absent in most cells and their phase locking reduced, and mice perform at chance.

In [ ]:
# What the files hold for the blocked claims (see the markdown above).
print("Files with a recording outside barrel cortex L4 (thalamic pipette multiunits of Fig. 6b,c would be):",
      int((meta.region != "barrel cortex").sum() + (meta.layer != "4").sum()), "of", len(meta), "L4 files; VPM files with light:",
      sum(r["has_light"] for r in vpm))
img = [p for r in recs for p in ([] if r["kind"] == "VPM" else r["general_fields"]) if "image" in p.lower()]
print("Files with an image or histology group under general/ (rabies tracing, Supp. Fig. 10):", len(img))
held = mode[(mode.from_trace == "whole-cell") & (mode.trials_near_0mV > 0)]
print("Whole-cell cells with trials held near 0 mV (the 6 E cells of Fig. 3 would be):", len(held), held[["path", "line", "trials_near_0mV"]]
      .assign(path=lambda d: d.path.str.split("/").str[1]).to_dict("records"), "; its class here:",
      cells.set_index("path").cls.get(held.path.iloc[0]) if len(held) else None, "; Supplementary Table 2 gives the one PV-ChR2 cell as FS")
# Descriptive only, not a test: the cells of the flagged mice, which may be among the ION-cut mice
fl = meas[meas.subject.isin(flagged) & (meas.cls != "VPM")]
fv = vm0[vm0.index.isin(fl.path)]
print(f"\nNot a test. Flagged mice {flagged}: {int((fl.cls == 'E').sum())} E cells ({int(((fl.cls == 'E') & (fl['mode'] == 'whole-cell')).sum())} whole-cell), "
      f"{int((fl.cls == 'FS').sum())} FS cells with a principal whisker. Whole-cell touch PSP amplitude mean {fv.amp_mv.mean():.2f} mV "
      f"(other E cells {vm0[~vm0.index.isin(fl.path) & vm0.index.isin(group('E').path)].amp_mv.mean():.2f}); whisking change (b) mean "
      f"{fv.whisk_epoch_change.mean():.2f} mV; Vm phase coupling mean {fv.vm_phase_strength.mean():.2f} mV")
ff = fl[fl.cls == "FS"]
if len(ff):
    print(f"   FS cells: spikes per touch mean {ff.spt.mean():.2f}; nonwhisking {ff['filtfilt:rate_nonwhisk'].mean():.2f}, whisking "
          f"{ff['filtfilt:rate_whisk'].mean():.2f} spikes/s; r2 {ff['filtfilt:r2'].round(2).tolist()}")

**Result: blocked (data missing): `exc-inhibition-timing` and `exc-inhibition-whisking`.** Only one
whole-cell file has trials held near 0 mV, 13 trials of the PV-ChR2 cell, which Supplementary Table 2
counts as FS; the 6 excitatory neurons recorded at rest and with depolarizing current are not in the
dandiset.

**Result: blocked (data missing): `vpm-photoinhibition`.** No file holds a thalamic recording made
with a pipette during light: the 130 single-cell files are all in barrel cortex layer 4, and the 18
VPM silicon probe files have no photostimulation series.

**Result: blocked (data missing): `rabies-vpm-input`.** The dandiset holds no histology or image data.

**Result: blocked (metadata insufficient): the seven ION-cut claims.** No file marks the ION-cut
mice. The 4 flagged mice are consistent with them (their whole-cell excitatory cells have touch PSPs
of 0.35 mV on average against 5.53 mV in the others, and their 2 FS cells 0.09 spikes per touch), but
they were found by these very properties, so they cannot test the claims.

## Summary

| Claim | Outcome | Earlier outcome (`000005-yu-2016-01`) |
|---|---|---|
| `behavior-touches` | partially reproduced | |
| `vpm-touch` | not reproduced | reproduced |
| `vpm-whisking` | partially reproduced | reproduced |
| `vpm-whisking-phase` | reproduced | |
| `exc-vm-touch-whisking` | partially reproduced | |
| `exc-threshold-touch-vs-whisking` | partially reproduced | |
| `exc-effective-reversal` | reproduced | |
| `exc-inhibition-timing` | blocked (data missing) | |
| `exc-inhibition-whisking` | blocked (data missing) | |
| `fs-touch-response` | partially reproduced | |
| `fs-vm-touch-whisking` | not reproduced | |
| `fs-whisking-rate` | partially reproduced | |
| `fs-whisking-phase` | reproduced | |
| `fs-touch-and-whisking` | reproduced | reproduced |
| `exc-mainly-touch` | not reproduced | not reproduced |
| `vpm-photoinhibition` | blocked (data missing) | |
| `exc-thalamus-photoinhibition` | partially reproduced | |
| `thalamus-drives-fs` | blocked (other: too few cells) | reproduced |
| `rabies-vpm-input` | blocked (data missing) | |
| `ion-cut-exc-no-touch` | blocked (metadata insufficient) | |
| `ion-cut-exc-whisking` | blocked (metadata insufficient) | |
| `ion-cut-exc-phase` | blocked (metadata insufficient) | |
| `ion-cut-fs-no-touch` | blocked (metadata insufficient) | |
| `sensory-input-drives-fs` | blocked (metadata insufficient) | blocked (metadata insufficient) |
| `ion-cut-fs-phase` | blocked (metadata insufficient) | |
| `ion-cut-behavior` | blocked (metadata insufficient) | |
| `pv-enphr-suppression` | partially reproduced | |
| `fs-suppress-movement-spiking` | partially reproduced | reproduced |
| `fs-enhance-touch-selectivity` | partially reproduced | not reproduced |
| `exc-touch-spikes-enphr` | not reproduced | |
| `auc-weak-strong` | reproduced | |
| `auc-groups` | blocked (methods underspecified) | |
| `behavior-false-alarms` | blocked (methods underspecified) | |

The eight claims that keep their ids have new criteria, written from the full text, so their
outcomes are not the same test as before; the attempt record says for each why the outcome differs.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The files name no cell type and do not mark the ION-cut
mice, so the cell classes are built by the claims file's rules (light tag, the waveform boundary read
approximately from Supplementary Fig. 6, PV inactivation, spike width) and every group includes the
cells of the ION-cut mice, which the paper left out of its control groups. The rules give 20 FS cells
(17 with a principal whisker) against the paper's 26 control FS cells, and 102 excitatory cells (97
measured) against 85; 8 light-driven cells fall outside the boundary as measured here. Only 3 of the
paper's 4 PV cells can be found. The VPM files name one principal whisker per file. Spike detection
and the spike-measure windows that the claims file leaves open are standard choices stated in the
helper module; the latency resolution, the bin position, the light-driven test, the waveform
measures and the chi-squared correction are asked in `questions/000005.yaml`.

What was found about the dandiset. Every finding of the earlier attempt is still present on the same
version: no cell types, no ION-cut marks, missing or partial genotypes, layer 4 for every cell,
undescribed optogenetic conditions, the wrong recording mode in 24 session descriptions, a
first-touch time divided by 1000, a sampling rate stated only by the timestamps, files without a
principal whisker and one principal whisker per VPM file, and doubtful light wavelengths. New here:
the injected current is stored in nA under a conversion to amperes that assumes uA; the
depolarized recordings of Fig. 3, the thalamic multiunits of Fig. 6b,c and the histology of
Supplementary Fig. 10 are not in the dandiset; trains at the VPM site run at 50 Hz in 12 files where
the Methods give 100 Hz; and the no-lick trial type is spelled two ways.